In [ ]:
import os
# Change the current working directory to your Google Drive folder
google_drive_path = "/content/drive/MyDrive/Earnings"
os.chdir(google_drive_path)

# Youorig_pre_df, post_df = process_phh_zip("phh_hands_backup.zip", True) can now call your function using just the filename

# --- To verify (optional) ---
# You can print the current working directory to confirm
print(f"Current working directory is: {os.getcwd()}")

os.environ["FINNHUB_API_KEY"] = 'd612jipr01qjrrugn72gd612jipr01qjrrugn730'

API_KEY = os.environ["FINNHUB_API_KEY"]

Current working directory is: /content/drive/MyDrive/Earnings


In [ ]:
!pip install vectorbt
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.7/420.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 6.4 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

class PurgedTimeSeriesSplit:
    def __init__(self, dates, n_splits=5, gap=0, window_size=None):
        self.dates = pd.to_datetime(dates).reset_index(drop=True)
        self.n_splits = n_splits
        self.gap = pd.Timedelta(days=gap) if isinstance(gap, int) else gap
        self.window_size = (
            pd.Timedelta(days=window_size) if isinstance(window_size, int)
            else window_size
        )

        self.unique_dates = pd.Index(self.dates.unique()).sort_values()

    def split(self, X, y=None, groups=None):

        U = self.unique_dates
        cut_points = np.linspace(
            0, len(U), self.n_splits + 2, dtype=int
        )[1:-1]

        for i, cp in enumerate(cut_points):

            train_cutoff = U[cp]

            if self.window_size is not None:
                train_start = train_cutoff - self.window_size
            else:
                train_start = U[0]

            test_start = train_cutoff + self.gap

            if i + 1 < len(cut_points):
                test_end = U[cut_points[i + 1]]
            else:
                test_end = U[-1]

            tr = self.dates[
                (self.dates >= train_start) &
                (self.dates <= train_cutoff)
            ].index.values

            te = self.dates[
                (self.dates >= test_start) &
                (self.dates <= test_end)
            ].index.values

            if len(tr) and len(te):
                yield tr, te

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from collections.abc import Callable

def rolling_window_classifier_consistency_check(
    df,
    target_column,
    time_column,
    window_size=3,
    step_size=1,
    threshold=0.5, # Default AUC threshold usually 0.5 (random)
    weight_scheme='exponential',
    weight_param=0.2,
    *,
    time_weight_map=None,
    normalize_time_weights=False
):
    """
    Check time consistency for each feature using a rolling window approach
    using Logistic Regression and ROC AUC.
    """
    feature_columns = [c for c in df.columns if c not in [target_column, time_column]]
    df_sorted = df.sort_values(by=time_column)
    time_points = df_sorted[time_column].unique()

    def _calc_weights_from_scheme(time_values, scheme='exponential', param=0.2):
        unique_times = np.unique(time_values)
        time_to_index = {t: i for i, t in enumerate(unique_times)}
        idx = np.array([time_to_index[t] for t in time_values])

        if scheme == 'none':
            w = np.ones(len(time_values), dtype=float)
        elif scheme == 'linear':
            w = 1.0 + idx * float(param)
        elif scheme == 'exponential':
            w = np.exp(idx * float(param))
        else:
            raise ValueError(f"Unknown weight_scheme: {scheme}")
        return w

    def _calc_weights_from_map(time_values, twm):
        if isinstance(twm, (dict, pd.Series)):
            w = np.array([float(twm.get(t, 1.0)) for t in time_values], dtype=float)
        elif isinstance(twm, Callable):
            w = np.array([float(twm(t)) for t in time_values], dtype=float)
        else:
            raise TypeError("time_weight_map must be dict, pandas Series, or callable")
        return w

    results = []

    for feature in tqdm(feature_columns, desc="Checking features"):
        feature_results = []

        for i in range(0, len(time_points) - window_size, step_size):
            if i + window_size >= len(time_points):
                break

            train_time = time_points[i : i + window_size]
            val_time = time_points[i + window_size]

            train_data = df_sorted[df_sorted[time_column].isin(train_time)]
            X_train = train_data[[feature]]
            y_train = train_data[target_column].to_numpy()

            # Skip if only one class is present in training
            if len(np.unique(y_train)) < 2:
                continue

            # --- sample weights ---
            if time_weight_map is not None:
                sample_weights = _calc_weights_from_map(train_data[time_column].values, time_weight_map)
            else:
                sample_weights = _calc_weights_from_scheme(train_data[time_column].values, weight_scheme, weight_param)

            if normalize_time_weights and sample_weights.size > 0:
                m = sample_weights.mean()
                if m > 0:
                    sample_weights = sample_weights / m

            val_data = df_sorted[df_sorted[time_column] == val_time]
            X_val = val_data[[feature]]
            y_val = val_data[target_column].to_numpy()

            # Fit Logistic Regression
            model = LogisticRegression(solver='liblinear')
            model.fit(X_train, y_train, sample_weight=sample_weights)

            # Train ROC AUC
            y_prob_train = model.predict_proba(X_train)[:, 1]
            try:
                train_auc = roc_auc_score(y_train, y_prob_train, sample_weight=sample_weights)
            except ValueError:
                train_auc = np.nan

            # Validation ROC AUC
            y_prob_val = model.predict_proba(X_val)[:, 1]
            try:
                # Validation is usually unweighted unless specified
                val_auc = roc_auc_score(y_val, y_prob_val) if len(np.unique(y_val)) > 1 else np.nan
            except ValueError:
                val_auc = np.nan

            feature_results.append({
                'Feature': feature,
                'Train_Time': train_time,
                'Val_Time': val_time,
                'Train_AUC': train_auc,
                'Val_AUC': val_auc
            })

        # Summaries
        if feature_results:
            val_auc_scores = [res['Val_AUC'] for res in feature_results if np.isfinite(res['Val_AUC'])]
            if len(val_auc_scores) > 0:
                mean_val_auc = float(np.mean(val_auc_scores))
                std_val_auc  = float(np.std(val_auc_scores))
                recent_chunk = val_auc_scores[-3:] if len(val_auc_scores) >= 3 else val_auc_scores
                recent_min_auc = float(np.min(recent_chunk))
                max_val_auc = float(np.max(val_auc_scores))

                is_consistent = (mean_val_auc > threshold) and (recent_min_auc > threshold)

                results.append({
                    'Feature': feature,
                    'Mean_Val_AUC': mean_val_auc,
                    'Std_Val_AUC': std_val_auc,
                    'Max_Val_AUC': max_val_auc,
                    'Recent_Min_AUC': recent_min_auc,
                    'Is_Consistent': is_consistent
                })

    return pd.DataFrame(results)

from lightgbm import LGBMClassifier

def cvs(X,y,model=LGBMClassifier(verbose=-1),std=False,return_scores=False):

  X=X.replace({np.inf: np.nan,-np.inf: np.nan})
  cv=PurgedTimeSeriesSplit(dates=pd.Series(X.index.get_level_values('earnings_ts')),gap=121)

  X=X.sort_index(level='earnings_ts')
  y=y.loc[X.index]

  scores=cross_val_score(model,X.fillna(-999),y,scoring='average_precision',error_score='raise',cv=cv)#np.mean(

  if return_scores:
    return scores

  if std:
    return scores.mean()/scores.std()

  return scores.mean()

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import requests

# 1. Setup Dates
tz = ZoneInfo("America/New_York")
today = datetime.now(tz).date()
tomorrow = today + timedelta(days=1)

url = "https://finnhub.io/api/v1/calendar/earnings"
# Expanding range to include tomorrow's pre-market
params = {"from": today.isoformat(), "to": tomorrow.isoformat(), "token": API_KEY}

data = requests.get(url, params=params).json()
rows = data.get("earningsCalendar", [])

# 2. Filter Logic
# AMC: After Market Close TODAY
after_close_today = [
    r["symbol"] for r in rows
    if r.get("date") == today.isoformat() and (r.get("hour") or "").lower() == "amc"
]

# BMO: Before Market Open TOMORROW
pre_market_tomorrow = [
    r["symbol"] for r in rows
    if r.get("date") == tomorrow.isoformat() and (r.get("hour") or "").lower() == "bmo"
]

earnings_tickers=after_close_today+pre_market_tomorrow
print(len(earnings_tickers))

105


In [ ]:
import pandas as pd
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import requests

def get_days_to_earnings(tickers, api_key, window_days=30):
    """
    Returns a Series showing days until (positive) or since (negative) earnings.
    """
    tz = ZoneInfo("America/New_York")
    today = datetime.now(tz).date()

    # Define a range to search (e.g., 30 days back to 30 days forward)
    start_date = (today - timedelta(days=window_days)).isoformat()
    end_date = (today + timedelta(days=window_days)).isoformat()

    url = "https://finnhub.io/api/v1/calendar/earnings"
    params = {"from": start_date, "to": end_date, "token": api_key}

    try:
        data = requests.get(url, params=params).json()
        rows = data.get("earningsCalendar", [])
    except Exception as e:
        print(f"Error fetching data: {e}")
        return pd.Series(dtype=int)

    # Map symbols to their dates
    # Note: If a ticker appears twice in the window, this keeps the most recent one found
    date_map = {
        r["symbol"]: datetime.strptime(r["date"], "%Y-%m-%d").date()
        for r in rows if r["symbol"] in tickers
    }

    # Calculate the difference in days
    results = {}
    for ticker in tickers:
        earnings_date = date_map.get(ticker)
        if earnings_date:
            delta = (earnings_date - today).days
            results[ticker] = delta
        else:
            results[ticker] = None # Or np.nan if you prefer

    return pd.Series(results, name="days_to_earnings")

# --- Example Usage ---
# API_KEY = "your_key_here"
# my_tickers = ["AAPL", "TSLA", "MSFT", "NVDA"]
# earnings_series = get_days_to_earnings(my_tickers, API_KEY)
# print(earnings_series)

In [ ]:
import numpy as np
import pandas as pd

def add_earnings_edge_features(
    df: pd.DataFrame,
    close: pd.Series,
    ret: pd.Series,
    vol: pd.Series | None = None,
    rets_panel: pd.DataFrame | None = None,      # your `rets` DataFrame (all tickers)
    earnings_ts: np.ndarray | None = None,       # output of _extract_earnings_ts(t)
    *,
    market_ticker: str = "SPY",
    vix_ticker: str = "^VIX",
    trend_windows: tuple[int, ...] = (20, 60, 252),
    hl_windows: tuple[int, ...] = (20, 60, 252),
    stat_windows: tuple[int, ...] = (20, 60),
    vwap_windows: tuple[int, ...] = (20, 60),
    eps: float = 1e-12,
) -> pd.DataFrame:
    """
    Adds extra features into df (aligned on df.index), no forward leakage for "earnings memory":
      - rolling log-price linear trend slope + R^2 (vectorized via convolution)
      - drawdown / run-up vs rolling max/min
      - return shape (skew/kurt), autocorr(1), downside realized vol
      - realized vol regime (rv_5/10/60 and ratios vs 20d)
      - liquidity/impact: dollar volume, dollar-volume z, Amihud illiquidity
      - OBV pressure and OBV z-score
      - rolling VWAP (price*vol / vol) deviation
      - short-horizon corr/beta vs SPY (and VIX corr if present)
      - earnings memory: previous earnings reactions, trailing mean/std/winrate, trading-days since prev earnings
    """

    idx = df.index
    close = close.reindex(idx).astype(float)
    ret = ret.reindex(idx).astype(float)

    # ---------- helpers ----------
    def _rolling_linreg_slope_r2(y: np.ndarray, w: int):
        """
        y: 1D array of floats (may contain NaN)
        returns slope and r2 arrays aligned to y length, where value appears at window end (t),
        i.e., first non-NaN at t = w-1
        """
        n = y.size
        slope = np.full(n, np.nan)
        r2 = np.full(n, np.nan)
        if n < w:
            return slope, r2

        x = np.arange(w, dtype=float)
        sum_x = x.sum()
        sum_x2 = (x * x).sum()
        denom = (w * sum_x2 - sum_x * sum_x)

        ones = np.ones(w, dtype=float)
        wts_rev = x[::-1].astype(float)

        y0 = np.nan_to_num(y, nan=0.0)
        finite = np.isfinite(y).astype(float)

        cnt = np.convolve(finite, ones, mode="valid")
        sum_y = np.convolve(y0, ones, mode="valid")
        sum_y2 = np.convolve(y0 * y0, ones, mode="valid")
        sum_xy = np.convolve(y0, wts_rev, mode="valid")

        ok = cnt == w
        b = np.full(sum_y.shape, np.nan)
        a = np.full(sum_y.shape, np.nan)

        b[ok] = (w * sum_xy[ok] - sum_x * sum_y[ok]) / (denom + eps)
        a[ok] = (sum_y[ok] - b[ok] * sum_x) / w

        ss_tot = np.full(sum_y.shape, np.nan)
        ss_res = np.full(sum_y.shape, np.nan)

        ss_tot[ok] = sum_y2[ok] - (sum_y[ok] * sum_y[ok]) / w
        ss_res[ok] = (
            sum_y2[ok]
            + w * (a[ok] * a[ok])
            + (b[ok] * b[ok]) * sum_x2
            - 2.0 * a[ok] * sum_y[ok]
            - 2.0 * b[ok] * sum_xy[ok]
            + 2.0 * a[ok] * b[ok] * sum_x
        )

        rr = 1.0 - (ss_res / (ss_tot + eps))
        rr = np.clip(rr, -1.0, 1.0)

        slope[w - 1 :] = b
        r2[w - 1 :] = rr
        return slope, r2

    # ---------- calendar / seasonality (numerical) ----------
    # (useful if your event-days skew by weekday/month, and it's not in your existing feature set)
    dow = idx.dayofweek.to_numpy(dtype=float)  # Mon=0
    mon = idx.month.to_numpy(dtype=float)      # 1..12
    df["dow_sin"] = np.sin(2 * np.pi * dow / 5.0)
    df["dow_cos"] = np.cos(2 * np.pi * dow / 5.0)
    df["month_sin"] = np.sin(2 * np.pi * (mon - 1.0) / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * (mon - 1.0) / 12.0)

    # ---------- trend fit on log-price ----------
    lp = np.log(close.replace(0.0, np.nan))
    y = lp.to_numpy(dtype=float)

    for w in trend_windows:
        sl, rr = _rolling_linreg_slope_r2(y, w)
        df[f"trend_lp_slope_{w}"] = sl                        # log-price per day
        df[f"trend_lp_r2_{w}"] = rr                           # fit quality
        df[f"trend_ann_{w}"] = np.exp(sl * 252.0) - 1.0       # implied annualized drift

    # ---------- drawdown / run-up vs rolling extrema ----------
    for w in hl_windows:
        roll_max = close.rolling(w).max()
        roll_min = close.rolling(w).min()
        df[f"dd_{w}"] = (close / (roll_max + eps)) - 1.0      # <= 0
        df[f"ru_{w}"] = (close / (roll_min + eps)) - 1.0      # >= 0

    # ---------- return distribution + dependence ----------
    for w in stat_windows:
        df[f"ret_skew_{w}"] = ret.rolling(w).skew()
        df[f"ret_kurt_{w}"] = ret.rolling(w).kurt()
        df[f"ret_autocorr1_{w}"] = ret.rolling(w).corr(ret.shift(1))
        neg = ret.clip(upper=0.0)
        df[f"downside_rv_{w}"] = np.sqrt((neg * neg).rolling(w).mean())

    # ---------- realized vol regime (short vs long) ----------
    rv_5 = ret.rolling(5).std()
    rv_10 = ret.rolling(10).std()
    rv_60 = ret.rolling(60).std()
    df["rv_5"] = rv_5
    df["rv_10"] = rv_10
    df["rv_60"] = rv_60

    # use your existing vol_20 if present; otherwise compute
    rv_20 = df["vol_20"] if "vol_20" in df.columns else ret.rolling(20).std()
    df["rv_5_over_20"] = rv_5 / (rv_20 + eps)
    df["rv_10_over_60"] = rv_10 / (rv_60 + eps)

    # ---------- volume / liquidity / impact (if volume provided) ----------
    if vol is not None:
        vol = vol.reindex(idx).astype(float)
        dvol = (close * vol).replace([np.inf, -np.inf], np.nan)
        df["dollar_vol"] = dvol

        # dollar-volume z-score (liquidity regime)
        dv_m = dvol.rolling(20).mean()
        dv_s = dvol.rolling(20).std()
        df["dollar_vol_z_20"] = (dvol - dv_m) / (dv_s + eps)

        # Amihud illiquidity (price impact proxy)
        # abs(ret) / dollar_vol, rolling mean; scaled for nicer magnitudes
        ill = (ret.abs() / (dvol + eps))
        df["amihud_20"] = ill.rolling(20).mean() * 1e6
        df["amihud_60"] = ill.rolling(60).mean() * 1e6

        # OBV + OBV z-score (volume pressure with direction)
        obv = (np.sign(ret.fillna(0.0)) * vol.fillna(0.0)).cumsum()
        df["obv"] = obv
        obv_m = obv.rolling(60).mean()
        obv_s = obv.rolling(60).std()
        df["obv_z_60"] = (obv - obv_m) / (obv_s + eps)

        # rolling VWAP proxy + deviation (different from your earnings-anchored VWAP)
        for w in vwap_windows:
            pv = (close * vol).rolling(w).sum()
            vv = vol.rolling(w).sum()
            rvwap = pv / (vv + eps)
            df[f"rvwap_{w}"] = rvwap
            df[f"px_vs_rvwap_{w}"] = (close / (rvwap + eps)) - 1.0

    # ---------- short-horizon market linkage (vs SPY) ----------
    if rets_panel is not None and market_ticker in rets_panel.columns:
        mret = rets_panel[market_ticker].reindex(idx).astype(float)
        df["corr_20_vs_spy"] = ret.rolling(20).corr(mret)

        # beta_20 (you already have beta_60 via anchors; this adds a shorter regime beta)
        cov20 = ret.rolling(20).cov(mret)
        var20 = mret.rolling(20).var()
        df["beta_20_vs_spy"] = cov20 / (var20 + eps)

    # (optional) “risk-off” linkage: corr to VIX changes (if present)
    if rets_panel is not None and vix_ticker in rets_panel.columns:
        vret = rets_panel[vix_ticker].reindex(idx).astype(float)
        df["corr_20_vs_vix"] = ret.rolling(20).corr(vret)

    # ---------- earnings memory (NO leakage) ----------
    if earnings_ts is not None and len(earnings_ts) > 0:
        trade_days = idx.values.astype("datetime64[ns]")
        ts = pd.to_datetime(pd.Series(earnings_ts)).values.astype("datetime64[ns]")

        pos_e = np.searchsorted(trade_days, ts, side="right") - 1
        pos_e = pos_e[(pos_e >= 0) & (pos_e < trade_days.size)]
        pos_e = np.unique(pos_e)

        if pos_e.size > 0:
            # trading-days since previous earnings day (strictly before current day)
            pos_series = pd.Series(np.nan, index=idx)
            pos_series.iloc[pos_e] = pos_e.astype(float)
            prev_pos = pos_series.ffill().shift(1)
            df["tdays_since_prev_ea"] = np.arange(len(idx), dtype=float) - prev_pos.to_numpy(dtype=float)

            # realized reaction of each earnings (after-close -> measured from event-day close)
            react1 = close.shift(-1) / close - 1.0
            react5 = close.shift(-5) / close - 1.0
            r1_e = react1.iloc[pos_e].to_numpy(dtype=float)
            r5_e = react5.iloc[pos_e].to_numpy(dtype=float)

            # trailing stats across earnings events (computed at event granularity)
            s_r1 = pd.Series(r1_e)
            mean4 = s_r1.rolling(4, min_periods=1).mean().to_numpy()
            std8 = s_r1.rolling(8, min_periods=1).std().to_numpy()
            win8 = s_r1.rolling(8, min_periods=1).apply(lambda a: np.mean(a > 0.0), raw=True).to_numpy()

            # write these into daily index as step functions that update the DAY AFTER earnings
            prev_r1 = pd.Series(np.nan, index=idx)
            prev_r5 = pd.Series(np.nan, index=idx)
            tr_mean4 = pd.Series(np.nan, index=idx)
            tr_std8 = pd.Series(np.nan, index=idx)
            tr_win8 = pd.Series(np.nan, index=idx)

            for k, p in enumerate(pos_e):
                j = p + 1  # start using this info AFTER the reaction is known (next day) -> no leakage
                if j < len(idx):
                    prev_r1.iloc[j] = r1_e[k]
                    prev_r5.iloc[j] = r5_e[k]
                    tr_mean4.iloc[j] = mean4[k]
                    tr_std8.iloc[j] = std8[k]
                    tr_win8.iloc[j] = win8[k]

            df["prev_ea_react_1d"] = prev_r1.ffill()
            df["prev_ea_react_5d"] = prev_r5.ffill()
            df["ea_trailing_mean_react1_4"] = tr_mean4.ffill()
            df["ea_trailing_std_react1_8"] = tr_std8.ffill()
            df["ea_trailing_winrate_8"] = tr_win8.ffill()

    return df

import pandas as pd
import yfinance as yf

def _download_ohlc(tickers, start, end=None):
    # sanitize tickers
    tickers = [t for t in tickers if isinstance(t, str) and t.strip()]
    tickers = list(dict.fromkeys(tickers))
    if not tickers:
        raise ValueError("No valid string tickers found.")

    frames = {}
    for t in tickers:
        df = yf.download(
            t,
            start=start, end=end,
            interval="1d",
            auto_adjust=False,
            progress=False,
            ignore_tz=True,
            group_by="column",
            threads=False,
        )
        if df is None or df.empty:
            continue

        # force tz-naive
        idx = pd.to_datetime(df.index, errors="coerce")
        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_convert(None)
        df = df.copy()
        df.index = idx
        df = df[~df.index.isna()].sort_index()

        frames[t] = df

    if not frames:
        raise ValueError("yfinance returned no data for any ticker.")

    # build panel
    panel = pd.concat(frames, axis=1)  # columns MultiIndex: (ticker, field)

    # pick close field
    fields = panel.columns.get_level_values(1).unique().tolist()
    use_close = "Adj Close" if "Adj Close" in fields else "Close"

    close = panel.xs(use_close, axis=1, level=1).sort_index().ffill(limit=3)
    high  = panel.xs("High", axis=1, level=1).sort_index().ffill(limit=3)
    low   = panel.xs("Low",  axis=1, level=1).sort_index().ffill(limit=3)

    return close, high, low




In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf

# ---------- TA helpers ----------
def rsi(series, period=14):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(alpha=1/period, adjust=False).mean()
    roll_down = down.ewm(alpha=1/period, adjust=False).mean()
    rs = roll_up / (roll_down + 1e-12)
    return 100 - (100 / (1 + rs))

def ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

def macd(series, fast=12, slow=26, signal=9):
    m = ema(series, fast) - ema(series, slow)
    s = ema(m, signal)
    h = m - s
    return m, s, h

def bollinger_z(series, window=20):
    ma = series.rolling(window).mean()
    sd = series.rolling(window).std()
    return (series - ma) / (sd + 1e-12)

def rolling_beta(x_ret, y_ret, window=60):
    cov = x_ret.rolling(window).cov(y_ret)
    var = y_ret.rolling(window).var()
    return cov / (var + 1e-12)

import numpy as np
import pandas as pd
import yfinance as yf

# ---------- Helpers ----------

def anchored_vwap(px: pd.Series, vol: pd.Series, anchor_mask: pd.Series) -> pd.Series:
    """AVWAP that resets on each True in anchor_mask (inclusive)."""
    px = px.astype(float)
    vol = vol.astype(float)
    anchor_mask = anchor_mask.reindex(px.index).fillna(False).astype(bool)

    # avoid divide-by-zero / NaN propagation
    v = vol.fillna(0.0)
    pv = (px.fillna(method="ffill") * v).fillna(0.0)

    seg = anchor_mask.cumsum()  # segment id increases at each anchor
    cum_pv = pv.groupby(seg).cumsum()
    cum_v = v.groupby(seg).cumsum()

    out = cum_pv / (cum_v.replace(0.0, np.nan))
    return out


def _extract_earnings_ts(ticker: str) -> np.ndarray:
    ed = yf.Ticker(ticker).earnings_dates
    if ed is None or ed.empty:
        return np.array([], dtype="datetime64[ns]")

    idx = ed.index  # tz-aware ET
    after_close = ed[idx.time >= pd.to_datetime("16:00").time()]

    return (
        after_close.index
        .tz_convert(None)
        .to_numpy(dtype="datetime64[ns]")
    )

# ---------- Core ----------
def build_earnings_reaction_dataset(
    tickers: list[str],
    horizon: int = 1,
    start: str = "2010-01-01",
    end: str | None = None,
    anchor: list[str] | None = ["SPY"],
    min_hist_days: int = 120,
    ma_windows: tuple[int, int, int] = (5, 20, 60),  # <-- 3 MAs
) -> pd.DataFrame:

    anchor = anchor or []
    all_tickers = list(dict.fromkeys(tickers + anchor))

    raw = yf.download(
        all_tickers,
        start=start, end=end,
        interval="1d",
        auto_adjust=False,
        group_by="ticker",
        threads=True,
        progress=False,
        ignore_tz=True,
    )

    # ---- prices panel ----
    if isinstance(raw.columns, pd.MultiIndex):
        fields = raw.columns.get_level_values(1).unique().tolist()
        use_px = "Adj Close" if "Adj Close" in fields else "Close"
        px = raw.xs(use_px, axis=1, level=1).sort_index()
        vol = raw.xs("Volume", axis=1, level=1).sort_index() if "Volume" in fields else None
    else:
        use_px = "Adj Close" if "Adj Close" in raw.columns else "Close"
        px = pd.DataFrame({all_tickers[0]: raw[use_px]}).sort_index()
        vol = pd.DataFrame({all_tickers[0]: raw["Volume"]}).sort_index() if "Volume" in raw.columns else None

    px   = raw.xs(use_px, axis=1, level=1).sort_index()
    opn  = raw.xs("Open", axis=1, level=1).sort_index()
    high = raw.xs("High", axis=1, level=1).sort_index()
    low  = raw.xs("Low", axis=1, level=1).sort_index()

    px = px.ffill(limit=3)
    if vol is not None:
        vol = vol.ffill(limit=3)

    rets = px.pct_change()
    logrets = np.log(px).diff()

    # Cache earnings timestamps per ticker (avoid double yfinance calls)
    earnings_ts = {}
    for t in tickers:
        try:
            earnings_ts[t] = _extract_earnings_ts(t)
        except Exception:
            earnings_ts[t] = np.array([], dtype="datetime64[ns]")

    # 3) Per-ticker daily features
    daily_feats = {}
    for t in tickers:
        if t not in px.columns:
            continue

        s = px[t]
        r = rets[t]
        lr = logrets[t]

        df = pd.DataFrame(index=px.index)
        df["px"] = s
        df["ret_1"] = r
        df["logret_1"] = lr
        df["ret_5"] = s.pct_change(5)
        df["vol_20"] = r.rolling(20).std()
        df["z_bb_20"] = bollinger_z(s, 20)

        df["rsi_14"] = rsi(s, 14)
        m, sig, hist = macd(s, 12, 26, 9)
        df["macd"] = m
        df["macd_signal"] = sig
        df["macd_hist"] = hist

        o = opn[t]
        h = high[t]
        l = low[t]
        c = s  # close

        rng = (h - l).replace(0, np.nan)

        # Intraday structure
        df["hl_range"] = rng / (c + 1e-12)
        df["body"] = (c - o) / (c + 1e-12)
        df["body_abs"] = (c - o).abs() / (c + 1e-12)

        df["close_loc"] = (c - l) / (rng + 1e-12)

        # Wicks
        df["upper_wick"] = (h - np.maximum(c, o)) / (c + 1e-12)
        df["lower_wick"] = (np.minimum(c, o) - l) / (c + 1e-12)

        # Gaps
        df["gap_open"] = o / c.shift(1) - 1.0

        # True Range / ATR proxy
        tr = np.maximum(
            h - l,
            np.maximum(
                (h - c.shift(1)).abs(),
                (l - c.shift(1)).abs()
            )
        )

        df["atr_14"] = tr.rolling(14).mean()
        df["atr_rel"] = df["atr_14"] / (c + 1e-12)


        df = add_regime_drift_features(
                df=df,
                o=o,
                h=h,
                l=l,
                c=c,
                r=r,
                vol_series=(vol[t] if vol is not None and t in vol.columns else None),
                anchors_rets=rets,
                anchor_list=anchor,
                add_percentiles=True,
                pct_window=252,
            )




        if "GLD" in rets.columns and "SLV" in rets.columns:
          g_ret = rets["GLD"]
          s_ret = rets["SLV"]

          df["gold_silver_ret_1"] = g_ret - s_ret
          df["gold_silver_ret_5"] = px["GLD"].pct_change(5) - px["SLV"].pct_change(5)
          df["gold_silver_corr_60"] = g_ret.rolling(60).corr(s_ret)

        # ---- volume features + AVWAP anchored to prior earnings ----
        if vol is not None and t in vol.columns:
            v = vol[t].astype(float)
            v_pct=vol.pct_change()[t].astype(float)
            df["vol"] = v
            df['vol_ret1']=v_pct
            for w in ma_windows:
                df[f"vol_ma_{w}"] = v.rolling(w).mean()
            df["vol_rel_20"] = df["vol"] / (df["vol_ma_20"] + 1e-12)


            # AVWAP anchor = most recent earnings trading day
            ts = earnings_ts.get(t)
            if ts is not None and ts.size:
                trade_days = df.index.values.astype("datetime64[ns]")
                pos = np.searchsorted(trade_days, ts, side="right") - 1
                pos = pos[(pos >= 0) & (pos < trade_days.size)]
                if pos.size:
                    anchor_mask = pd.Series(False, index=df.index)
                    anchor_mask.iloc[np.unique(pos)] = True
                    avwap = anchored_vwap(s, v, anchor_mask)
                    df["avwap_ea"] = avwap
                    df["px_vs_avwap_ea"] = (s / avwap) - 1.0

        # performance = rolling mean of daily returns
        for w in ma_windows:
            df[f"perf_ma_{w}"] = r.rolling(w).mean()

        # anchors
        for a in anchor:
            if a not in rets.columns:
                continue
            a_ret = rets[a]
            df[f"rel_ret_1_vs_{a}"] = r - a_ret
            df[f"rel_ret_5_vs_{a}"] = s.pct_change(5) - px[a].pct_change(5)
            df[f"corr_60_vs_{a}"] = r.rolling(60).corr(a_ret)
            df[f"beta_60_vs_{a}"] = rolling_beta(r, a_ret, 60)

        df["fwd_ret_h"] = s.shift(-horizon) / s - 1.0

        df = add_earnings_edge_features(
          df,
          close=s,
          ret=r,
          vol=(vol[t] if vol is not None and t in vol.columns else None),
          rets_panel=rets,
          earnings_ts=earnings_ts.get(t)  # in synthetic, pass ts from _extract_earnings_ts(t)
            )

        daily_feats[t] = df


    # 4) Earnings timestamps -> vectorized event mapping
    rows = []
    for t in tickers:
        feat_df = daily_feats.get(t)
        if feat_df is None:
            continue

        ts = earnings_ts.get(t)
        if ts is None or ts.size == 0:
            continue

        trade_days = feat_df.index.values.astype("datetime64[ns]")
        pos = np.searchsorted(trade_days, ts, side="right") - 1

        # filters
        ok = (pos >= min_hist_days) & (pos >= 0) & ((pos + horizon) < trade_days.size)
        if not ok.any():
            continue

        pos_ok = pos[ok]
        event_days = pd.to_datetime(trade_days[pos_ok])

        # label availability
        fwd = feat_df["fwd_ret_h"].iloc[pos_ok].to_numpy(dtype=float)
        ok2 = np.isfinite(fwd)


        if not ok2.any():
            continue

        pos_ok = pos_ok[ok2]
        event_days = event_days[ok2]
        ts_ok = pd.to_datetime(ts[ok][ok2])

        fwd = fwd[ok2]
        y_up = (fwd > 0).astype(int)

        X = feat_df.iloc[pos_ok].drop(columns=["fwd_ret_h"])

        out = X.copy()
        out.insert(0, "ticker", t)
        out.insert(1, "earnings_ts", ts_ok.to_numpy())
        out.insert(2, "event_day", event_days.to_numpy())
        out.insert(3, "horizon_days", horizon)
        out.insert(4, "y_up", y_up)
        out.insert(5, "fwd_ret_h", fwd)

        rows.append(out.reset_index(drop=True))

    if not rows:
        return pd.DataFrame()

    data = pd.concat(rows, ignore_index=True)
    data = data.sort_values(["ticker", "event_day"]).reset_index(drop=True)
    return data,px

# ---------- Synthetic "next earnings is today" test set ----------

def build_synthetic_earnings_test_dataset(
    tickers: list[str],
    asof: str | pd.Timestamp | None = None,   # "assumed earnings date"
    start: str = "2010-01-01",
    anchor: list[str] | None = ["SPY"],
    min_hist_days: int = 120,
    ma_windows: tuple[int, int, int] = (5, 20, 60),
) -> pd.DataFrame:
    """
    One row per ticker, representing features computed as-of the last trading day <= asof.
    Labels are unknown, so y_up and fwd_ret_h are NaN.
    """

    anchor = anchor or []
    all_tickers = list(dict.fromkeys(tickers + anchor))

    if asof is None:
        asof_ts = pd.Timestamp.today().normalize()
    else:
        asof_ts = pd.Timestamp(asof).tz_localize(None).normalize()

    # Download ONLY up to asof (end is exclusive in yfinance; add 1 day)
    end = (asof_ts + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    raw = yf.download(
        all_tickers,
        start=start, end=end,
        interval="1d",
        auto_adjust=False,
        group_by="ticker",
        threads=True,
        progress=False,
        ignore_tz=True,
    )

    # ---- prices panel ----
    if isinstance(raw.columns, pd.MultiIndex):
        fields = raw.columns.get_level_values(1).unique().tolist()
        use_px = "Adj Close" if "Adj Close" in fields else "Close"
        px = raw.xs(use_px, axis=1, level=1).sort_index()
        vol = raw.xs("Volume", axis=1, level=1).sort_index() if "Volume" in fields else None
    else:
        use_px = "Adj Close" if "Adj Close" in raw.columns else "Close"
        px = pd.DataFrame({all_tickers[0]: raw[use_px]}).sort_index()
        vol = pd.DataFrame({all_tickers[0]: raw["Volume"]}).sort_index() if "Volume" in raw.columns else None

    px   = raw.xs(use_px, axis=1, level=1).sort_index()
    opn  = raw.xs("Open", axis=1, level=1).sort_index()
    high = raw.xs("High", axis=1, level=1).sort_index()
    low  = raw.xs("Low", axis=1, level=1).sort_index()

    px = px.ffill(limit=3)
    if vol is not None:
        vol = vol.ffill(limit=3)

    rets = px.pct_change()
    logrets = np.log(px).diff()

    rows = []
    for t in tickers:
        if t not in px.columns:
            continue

        s = px[t]
        r = rets[t]
        lr = logrets[t]

        df = pd.DataFrame(index=px.index)
        df["px"] = s
        df["ret_1"] = r
        df["logret_1"] = lr
        df["ret_5"] = s.pct_change(5)
        df["vol_20"] = r.rolling(20).std()
        df["z_bb_20"] = bollinger_z(s, 20)

        df["rsi_14"] = rsi(s, 14)
        m, sig, hist = macd(s, 12, 26, 9)
        df["macd"] = m
        df["macd_signal"] = sig
        df["macd_hist"] = hist


        o = opn[t]
        h = high[t]
        l = low[t]
        c = s  # close

        rng = (h - l).replace(0, np.nan)

        # Intraday structure
        df["hl_range"] = rng / (c + 1e-12)
        df["body"] = (c - o) / (c + 1e-12)
        df["body_abs"] = (c - o).abs() / (c + 1e-12)

        df["close_loc"] = (c - l) / (rng + 1e-12)

        # Wicks
        df["upper_wick"] = (h - np.maximum(c, o)) / (c + 1e-12)
        df["lower_wick"] = (np.minimum(c, o) - l) / (c + 1e-12)

        # Gaps
        df["gap_open"] = o / c.shift(1) - 1.0

        # True Range / ATR proxy
        tr = np.maximum(
            h - l,
            np.maximum(
                (h - c.shift(1)).abs(),
                (l - c.shift(1)).abs()
            )
        )

        df["atr_14"] = tr.rolling(14).mean()
        df["atr_rel"] = df["atr_14"] / (c + 1e-12)

        '''
        df = add_regime_drift_features(
              df=df,
              o=o,
              h=h,
              l=l,
              c=c,
              r=r,
              vol_series=(vol[t] if vol is not None and t in vol.columns else None),
              anchors_rets=rets,
              anchor_list=anchor,
              add_percentiles=True,
              pct_window=252,
                )
        '''

        if "GLD" in rets.columns and "SLV" in rets.columns:
          g_ret = rets["GLD"]
          s_ret = rets["SLV"]

          df["gold_silver_ret_1"] = g_ret - s_ret
          df["gold_silver_ret_5"] = px["GLD"].pct_change(5) - px["SLV"].pct_change(5)
          df["gold_silver_corr_60"] = g_ret.rolling(60).corr(s_ret)



        # volume features + AVWAP anchored to prior earnings
        if vol is not None and t in vol.columns:
            v = vol[t].astype(float)
            df["vol"] = v


            v_pct=vol.pct_change()[t].astype(float)

            df['vol_ret1']=v_pct
            for w in ma_windows:
                df[f"vol_ma_{w}"] = v.rolling(w).mean()
            df["vol_rel_20"] = df["vol"] / (df["vol_ma_20"] + 1e-12)

            try:
                ts = _extract_earnings_ts(t)
            except Exception:
                ts = np.array([], dtype="datetime64[ns]")

            if ts.size:
                trade_days = df.index.values.astype("datetime64[ns]")
                pos_e = np.searchsorted(trade_days, ts, side="right") - 1
                pos_e = pos_e[(pos_e >= 0) & (pos_e < trade_days.size)]
                if pos_e.size:
                    anchor_mask = pd.Series(False, index=df.index)
                    anchor_mask.iloc[np.unique(pos_e)] = True
                    avwap = anchored_vwap(s, v, anchor_mask)
                    df["avwap_ea"] = avwap
                    df["px_vs_avwap_ea"] = (s / avwap) - 1.0

        # performance MAs
        for w in ma_windows:
            df[f"perf_ma_{w}"] = r.rolling(w).mean()

        # anchors
        for a in anchor:
            if a not in rets.columns:
                continue
            a_ret = rets[a]
            df[f"rel_ret_1_vs_{a}"] = r - a_ret
            df[f"rel_ret_5_vs_{a}"] = s.pct_change(5) - px[a].pct_change(5)
            df[f"corr_60_vs_{a}"] = r.rolling(60).corr(a_ret)
            df[f"beta_60_vs_{a}"] = rolling_beta(r, a_ret, 60)


                # synthetic: one assumed earnings timestamp for this ticker
        earnings_ts_arr = np.array([np.datetime64(asof_ts)], dtype="datetime64[ns]")

        df = add_earnings_edge_features(
            df,
            close=s,
            ret=r,
            vol=(vol[t] if vol is not None and t in vol.columns else None),
            rets_panel=rets,
            earnings_ts=earnings_ts_arr,     # ✅ no .get()
        )


        # pick the synthetic event day = last trading day <= asof
        trade_days = df.index.values.astype("datetime64[ns]")
        asof_np = np.datetime64(asof_ts.to_datetime64())
        pos = np.searchsorted(trade_days, asof_np, side="right") - 1
        if pos < 0 or pos < min_hist_days:
            continue

        event_day = pd.to_datetime(trade_days[pos])
        X = df.iloc[[pos]].copy()

        out = X.copy()
        out.insert(0, "ticker", t)
        out.insert(1, "earnings_ts", np.datetime64(asof_ts))
        out.insert(2, "event_day", np.datetime64(event_day))
        out.insert(3, "horizon_days", np.nan)
        out.insert(4, "y_up", np.nan)
        out.insert(5, "fwd_ret_h", np.nan)

        rows.append(out.reset_index(drop=True))

    if not rows:
        return pd.DataFrame()

    data = pd.concat(rows, ignore_index=True).sort_values(["ticker"]).reset_index(drop=True)
    return data


# ---- Example ----
SIZE_CAP_ETFS = ["VOO", "IJH", "IJR", "IWM"] # Core Market Caps
STYLE_ETFS    = ["VUG", "VTV", "VBK", "VBR"] # Growth vs Value
COUNTRY_ETFS  = ["EWJ","EWU","EWC","EWG","EWP","EWL","EWD","EIS","EEM","VEA","VXUS","IXUS"]
SECTOR_ETFS   = ["XLC","XLY","XLP","XLE","XLF","XLV","XLI","XLB","XLRE","XLK","XLU"]
OTHER_STOCKS  = ["SPY","^VIX",'GLD','SLV','BTCUSD']

ANCHORS = COUNTRY_ETFS + SECTOR_ETFS + OTHER_STOCKS+STYLE_ETFS+SIZE_CAP_ETFS

'''
ds,px= build_earnings_reaction_dataset(earnings_tickers, horizon=3, start="2025-01-05", anchor=ANCHORS)  ###HORIZON = 1

#is_friday = ds['earnings_ts'].dt.day_name() == 'Friday'
#ds=ds[~is_friday]
ticker_counts=ds[['ticker','earnings_ts']].value_counts()
bad_tickers=ticker_counts[ticker_counts>1].index.get_level_values('ticker').tolist()
ds=ds[~ds['ticker'].isin(bad_tickers)]

data=ds.copy()
data = data.sort_values(["event_day", "ticker"])
data.set_index(['ticker','earnings_ts'],inplace=True)
#data.set_index(META_COLS,inplace=True)

LABEL_COLS = ["y_up"]          # or ["y_up", "fwd_ret_h"] if you want both
META_COLS  = ["ticker", "earnings_ts"]

# anything forward-looking (keep this list explicit)
FUTURE_COLS = ["fwd_ret_h"]    # add others if you create them later (e.g. fwd_ret_1, fwd_ret_5, etc.)

DROP_FROM_X = set(LABEL_COLS + META_COLS + FUTURE_COLS+['event_day','horizon_days'])

X = data.drop(columns=[c for c in data.columns if c in DROP_FROM_X]).copy()
y = data["y_up"].astype(int).copy()

y = y.loc[data.index]
X = X.loc[y.index]

print(X.shape)
'''

'\nds,px= build_earnings_reaction_dataset(earnings_tickers, horizon=3, start="2025-01-05", anchor=ANCHORS)  ###HORIZON = 1\n\n#is_friday = ds[\'earnings_ts\'].dt.day_name() == \'Friday\'\n#ds=ds[~is_friday]\nticker_counts=ds[[\'ticker\',\'earnings_ts\']].value_counts()\nbad_tickers=ticker_counts[ticker_counts>1].index.get_level_values(\'ticker\').tolist()\nds=ds[~ds[\'ticker\'].isin(bad_tickers)]\n\ndata=ds.copy()\ndata = data.sort_values(["event_day", "ticker"])\ndata.set_index([\'ticker\',\'earnings_ts\'],inplace=True)\n#data.set_index(META_COLS,inplace=True)\n\nLABEL_COLS = ["y_up"]          # or ["y_up", "fwd_ret_h"] if you want both\nMETA_COLS  = ["ticker", "earnings_ts"]\n\n# anything forward-looking (keep this list explicit)\nFUTURE_COLS = ["fwd_ret_h"]    # add others if you create them later (e.g. fwd_ret_1, fwd_ret_5, etc.)\n\nDROP_FROM_X = set(LABEL_COLS + META_COLS + FUTURE_COLS+[\'event_day\',\'horizon_days\'])\n\nX = data.drop(columns=[c for c in data.columns if c in DROP

In [ ]:
ds1=pd.read_csv('ds_v1.csv',index_col='Unnamed: 0')
ds2=pd.read_csv('ds_v2.csv',index_col='Unnamed: 0')
ds=pd.concat([ds1,ds2])
#ds=new_events.copy()

#is_friday = ds['earnings_ts'].dt.day_name() == 'Friday'
#ds=ds[~is_friday]
ticker_counts=ds[['ticker','earnings_ts']].value_counts()
bad_tickers=ticker_counts[ticker_counts>1].index.get_level_values('ticker').tolist()
ds=ds[~ds['ticker'].isin(bad_tickers)]

to_drop=[
    "tdays_since_prev_ea",
    "prev_ea_react_1d",
    "prev_ea_react_5d",
    "ea_trailing_mean_react1_4",
    "ea_trailing_std_react1_8",
    "ea_trailing_winrate_8"
]


data=ds.copy().drop(to_drop,axis=1)
data = data.sort_values(["event_day", "ticker"])
data.set_index(['ticker','earnings_ts'],inplace=True)
#data.set_index(META_COLS,inplace=True)

LABEL_COLS = ["y_up"]          # or ["y_up", "fwd_ret_h"] if you want both
META_COLS  = ["ticker", "earnings_ts"]

# anything forward-looking (keep this list explicit)
FUTURE_COLS = ["fwd_ret_h"]    # add others if you create them later (e.g. fwd_ret_1, fwd_ret_5, etc.)

DROP_FROM_X = set(LABEL_COLS + META_COLS + FUTURE_COLS+['event_day','horizon_days'])

X = data.drop(columns=[c for c in data.columns if c in DROP_FROM_X]).copy()
y = data["y_up"].astype(int).copy()

y = y.loc[data.index]
X = X.loc[y.index]

print(X.shape)

(13865, 222)


In [ ]:
import yfinance as yf
import pandas as pd

def get_ohlcv(
    tickers,
    start="1998-01-01",
    end=None,
    use_adj_close=True,
    ffill_limit=3,
):
    """
    Returns:
        close, high, low, open_, volume   (all pd.DataFrame)

    Each:
        index   = trading days
        columns = tickers
    """

    raw = yf.download(
        tickers,
        start=start,
        end=end,
        interval="1d",
        auto_adjust=False,
        group_by="ticker",
        progress=False,
        threads=True,
        ignore_tz=True,
    )

    def _extract(field):
        if isinstance(raw.columns, pd.MultiIndex):
            if field not in raw.columns.get_level_values(1):
                return None
            df = raw.xs(field, axis=1, level=1)
        else:
            if field not in raw.columns:
                return None
            df = raw[[field]]
            df.columns = tickers[:1]

        df = df.sort_index()
        df = df.ffill(limit=ffill_limit)
        return df

    # close / adj close
    if use_adj_close and "Adj Close" in raw.columns.get_level_values(1):
        close = _extract("Adj Close")
    else:
        close = _extract("Close")

    open_  = _extract("Open")
    high   = _extract("High")
    low    = _extract("Low")
    volume = _extract("Volume")

    return close, high, low, open_, volume


import numpy as np
import pandas as pd

def derive_exit_labels_first_touch_approx(
    X: pd.DataFrame,
    open_df: pd.DataFrame,
    high_df: pd.DataFrame,
    low_df: pd.DataFrame,
    close_df: pd.DataFrame,
    horizon: int = 5,
    tp: float = 0.02,
    sl: float = 0.01,
    include_day1_intraday: bool = True,
    both_hit_rule: str = "sl_first",

    # --- NEW: volatility-based exits ---
    vol_mode: str = "fixed",          # "fixed" | "atr" | "close_vol"
    vol_lookback: int = 20,
    tp_vol_mult: float = 2.0,
    sl_vol_mult: float = 1.5,
    min_tp: float = 0.0,              # clamp (optional)
    min_sl: float = 0.0,
    max_tp: float = np.inf,
    max_sl: float = np.inf,
) -> pd.DataFrame:

    if not isinstance(X.index, pd.MultiIndex) or X.index.nlevels < 2:
        raise ValueError("X must have a MultiIndex like (ticker, earnings_ts).")
    if both_hit_rule not in {"sl_first", "tp_first", "skip"}:
        raise ValueError("both_hit_rule must be one of: 'sl_first', 'tp_first', 'skip'.")
    if vol_mode not in {"fixed", "atr", "close_vol"}:
        raise ValueError("vol_mode must be one of: 'fixed', 'atr', 'close_vol'.")

    tickers = X.index.get_level_values(0)
    earnings_ts = pd.to_datetime(X.index.get_level_values(1)).tz_localize(None)

    trade_days = close_df.index.values.astype("datetime64[ns]")
    ts_np = earnings_ts.to_numpy(dtype="datetime64[ns]")
    pos = np.searchsorted(trade_days, ts_np, side="right") - 1

    valid = (pos >= 0) & ((pos + horizon) < len(trade_days))
    event_day = pd.to_datetime(trade_days[np.clip(pos, 0, len(trade_days) - 1)])

    out = pd.DataFrame(index=X.index)
    out["event_day"] = event_day

    entry_px = []
    mfe_list = []
    mae_list = []
    exit_code_list = []
    exit_day_list = []
    tp_used_list = []
    sl_used_list = []
    vol_used_list = []

    # --- Precompute vol surfaces (fast + simple) ---
    # For ATR we need prev close; for close_vol we use rolling std of returns.
    if vol_mode == "atr":
        prev_close = close_df.shift(1)
        tr = pd.concat([
            (high_df - low_df),
            (high_df - prev_close).abs(),
            (low_df - prev_close).abs(),
        ], axis=0).groupby(level=0).max()  # (hacky) not correct because concat changes index
        # Better explicit TR:
        tr = np.maximum(high_df - low_df, np.maximum((high_df - prev_close).abs(), (low_df - prev_close).abs()))
        atr = tr.rolling(vol_lookback, min_periods=vol_lookback).mean()
        # Convert to % of price (use close as a stable denominator; entry close is even better per-trade)
        atr_pct = atr / close_df

    elif vol_mode == "close_vol":
        ret = close_df.pct_change()
        vol = ret.rolling(vol_lookback, min_periods=vol_lookback).std()
        vol_pct = vol  # already in return units (daily)

    for (tkr, ev_pos, ok) in zip(tickers, pos, valid):
        if (not ok) or (tkr not in close_df.columns):
            entry_px.append(np.nan)
            mfe_list.append(np.nan)
            mae_list.append(np.nan)
            exit_code_list.append(np.nan)
            exit_day_list.append(np.nan)
            tp_used_list.append(np.nan)
            sl_used_list.append(np.nan)
            vol_used_list.append(np.nan)
            continue

        entry = float(close_df.iloc[ev_pos][tkr])
        entry_px.append(entry)

        start = ev_pos + 1
        end = ev_pos + 1 + horizon  # exclusive

        # --- per-trade TP/SL ---
        if vol_mode == "fixed":
            tp_i, sl_i = float(tp), float(sl)
            vol_i = np.nan

        elif vol_mode == "atr":
            # use ATR% from event day (computed using prior lookback)
            vol_i = float(atr_pct.iloc[ev_pos][tkr])
            if not np.isfinite(vol_i) or vol_i <= 0:
                tp_i = sl_i = np.nan
            else:
                tp_i = np.clip(tp_vol_mult * vol_i, min_tp, max_tp)
                sl_i = np.clip(sl_vol_mult * vol_i, min_sl, max_sl)

        else:  # "close_vol"
            vol_i = float(vol_pct.iloc[ev_pos][tkr])
            if not np.isfinite(vol_i) or vol_i <= 0:
                tp_i = sl_i = np.nan
            else:
                tp_i = np.clip(tp_vol_mult * vol_i, min_tp, max_tp)
                sl_i = np.clip(sl_vol_mult * vol_i, min_sl, max_sl)

        tp_used_list.append(tp_i)
        sl_used_list.append(sl_i)
        vol_used_list.append(vol_i)

        # If we couldn't compute vol-based thresholds, mark invalid for this trade
        if not np.isfinite(tp_i) or not np.isfinite(sl_i):
            mfe_list.append(np.nan)
            mae_list.append(np.nan)
            exit_code_list.append(np.nan)
            exit_day_list.append(np.nan)
            continue

        # Diagnostics
        h_win = high_df.iloc[start:end][tkr].to_numpy(dtype=float)
        l_win = low_df.iloc[start:end][tkr].to_numpy(dtype=float)
        o_win = open_df.iloc[start:end][tkr].to_numpy(dtype=float)

        mfe = np.nanmax(h_win / entry - 1.0)
        mae = np.nanmin(np.minimum(o_win / entry - 1.0, l_win / entry - 1.0))

        mfe_list.append(mfe)
        mae_list.append(mae)

        # First-touch simulation
        exit_code = 0
        exit_day = np.nan

        for i, day_idx in enumerate(range(start, end), start=1):
            o = float(open_df.iloc[day_idx][tkr])
            o_ret = o / entry - 1.0

            if o_ret <= -sl_i:
                exit_code, exit_day = -1, i
                break
            if o_ret >= tp_i:
                exit_code, exit_day = 1, i
                break

            if (i == 1) and (not include_day1_intraday):
                continue

            h = float(high_df.iloc[day_idx][tkr])
            l = float(low_df.iloc[day_idx][tkr])

            touched_tp = (h / entry - 1.0) >= tp_i
            touched_sl = (l / entry - 1.0) <= -sl_i

            if touched_tp and not touched_sl:
                exit_code, exit_day = 1, i
                break
            if touched_sl and not touched_tp:
                exit_code, exit_day = -1, i
                break
            if touched_tp and touched_sl:
                if both_hit_rule == "sl_first":
                    exit_code, exit_day = -1, i
                elif both_hit_rule == "tp_first":
                    exit_code, exit_day = 1, i
                else:
                    exit_code, exit_day = np.nan, np.nan
                break

        exit_code_list.append(exit_code)
        exit_day_list.append(exit_day)

    out["entry_px"] = entry_px
    out["vol_used"] = vol_used_list
    out["tp_used"] = tp_used_list
    out["sl_used"] = sl_used_list
    out["MFE"] = mfe_list
    out["MAE"] = mae_list
    out["exit_code"] = exit_code_list
    out["exit_day"] = exit_day_list
    out["y_tp_first"] = np.where(valid, (out["exit_code"] == 1).astype(float), np.nan)
    return out

###Stitch Regime Features

[I 2026-03-04 06:53:03,698] A new study created in memory with name: no-name-3344b430-ad71-4c6f-9c62-c408b7a9a764
[W 2026-03-04 06:53:12,410] Trial 0 failed with parameters: {'horizon': 2, 'tp': 0.01785436060870726, 'sl': 0.01079006193334046} because of the following error: NameError("name 'meta_cvs' is not defined").
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_616/877677057.py", line 55, in objective
    score=meta_cvs(X_use, y_use, primary_fit,meta_model,primary_cvs=True,close=close,high=high,low=low,open=open_)
          ^^^^^^^^
NameError: name 'meta_cvs' is not defined
[W 2026-03-04 06:53:12,414] Trial 0 failed with value None.


NameError: name 'meta_cvs' is not defined

In [ ]:
import pandas as pd

def chronological_split(df, y,val_ratio=0.2, date_level='earnings_ts'):
    """
    Splits a MultiIndex DataFrame chronologically based on the date level.

    Args:
        df (pd.DataFrame): MultiIndex DF (e.g., ['ticker', 'date'])
        val_ratio (float): Proportion of data to use for validation (0 to 1)
        date_level (str): The name of the index level containing dates

    Returns:
        train_df, val_df: Two DataFrames split by date
    """
    # 1. Get unique sorted dates from the index
    unique_dates = df.index.get_level_values(date_level).unique().sort_values()

    # 2. Calculate the split index
    split_idx = int(len(unique_dates) * (1 - val_ratio))
    split_date = unique_dates[split_idx]

    # 3. Slice the dataframe
    # We use xs or index slicing to ensure we don't mix dates
    train_df = df.iloc[df.index.get_level_values(date_level) < split_date]
    val_df = df.iloc[df.index.get_level_values(date_level) >= split_date]

    y_train=y.loc[train_df.index]
    y_val=y.loc[val_df.index]

    return train_df, val_df,y_train,y_val


#close, high, low, open_, volume=get_ohlcv(x_test.index.get_level_values(0).tolist(),"2025-05-05")
x_train,x_test,y_train,y_test=chronological_split(X,y_new,val_ratio=0.1)
#sample_weights=compute_time_decay_weights(pd.to_datetime(x_train.index.get_level_values('earnings_ts')))

In [ ]:
from sklearn.metrics import average_precision_score as aps
model=LGBMClassifier(verbose=-1)

#model=stacking_model
model.fit(x_train.fillna(-999),y_train)
preds=model.predict_proba(x_test.fillna(-999))[:,-1]
#preds=model.predict(x_test.fillna(-999))

preds=pd.Series(preds,index=x_test.index)

In [ ]:
preds=model.predict(x_test.fillna(-999))
preds=pd.Series(preds,index=x_test.index)
preds.value_counts()

In [ ]:
close, high, low, open_, volume=get_ohlcv(x_test.index.get_level_values(0).tolist(),"2025-05-05")
ohlcv_dct={'high': high,'low': low,'close': close,'open': open_}

In [ ]:
# proba_last_class: Series with MultiIndex (ticker, earnings_ts)
# ohlcv: dict with DataFrames 'open','high','low','close' indexed by trading dates, columns tickers

pf = simulate_earnings_long_vbt(
    preds,
    ohlcv_dct,
    init_cash=100_000,
    stop_loss=best_params['sl'],
    take_profit=best_params['tp'],
    horizon=best_params['horizon'],
    min_proba=0.505,
    top_n=None,
    weighting="proba",
    fees=0.0005,
    slippage=0.0002,
)

In [ ]:
pf.plot().show()

In [ ]:
pf.stats()

In [ ]:
pf = simulate_earnings_bidir_vbt(
    preds,
    ohlcv_dct,
    init_cash=100_000,
    stop_loss=best_params['sl'],
    take_profit=best_params['tp'],
    horizon=best_params['horizon'],
    weighting="proba",
    min_proba_long=0.6,
    max_proba_short=0.495,
    fees=0.0005,
    slippage=0.0002,
)

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import vectorbt as vbt


def _dbg(msg: str, enabled: bool) -> None:
    if enabled:
        print(msg)


def _df_nan_report(df: pd.DataFrame, name: str, enabled: bool, max_cols: int = 10) -> None:
    if not enabled:
        return
    total = df.size
    n_nan = int(df.isna().to_numpy().sum())
    _dbg(f"[{name}] shape={df.shape}  nan={n_nan}/{total} ({(n_nan/total if total else 0):.2%})", enabled)
    if n_nan:
        cols = df.isna().sum().sort_values(ascending=False)
        cols = cols[cols > 0].head(max_cols)
        if len(cols):
            _dbg(f"[{name}] top NaN columns:\n{cols.to_string()}", enabled)


def _bool_report(b: pd.DataFrame, name: str, enabled: bool) -> None:
    if not enabled:
        return
    n_true = int(b.to_numpy().sum())
    _dbg(f"[{name}] true={n_true}  days_with_any={int(b.any(axis=1).sum())}", enabled)


def _row_sum_report(df: pd.DataFrame, name: str, enabled: bool) -> None:
    if not enabled:
        return
    s = df.sum(axis=1)
    _dbg(
        f"[{name}] row_sum: min={float(s.min()):.6f}  max={float(s.max()):.6f}  "
        f"mean={float(s.mean()):.6f}  nonzero_days={int((s>0).sum())}",
        enabled,
    )
    bad = (s > 1.000001) | (s < -1e-9)
    if bad.any():
        _dbg(f"[{name}] WARNING: row_sum outside [0,1] on {int(bad.sum())} days (showing first 5):\n{s[bad].head()}", enabled)


def simulate_earnings_bidir_vbt(
    proba_last_class: pd.Series,
    ohlcv: dict[str, pd.DataFrame],
    *,
    init_cash: float = 100_000.0,
    stop_loss: float | None = 0.05,
    take_profit: float | None = 0.10,
    horizon: int = 5,
    min_proba_long: float | None = None,
    top_n_long: int | None = None,
    max_proba_short: float | None = None,
    top_n_short: int | None = None,
    weighting: str = "proba",  # "proba" or "equal"
    fees: float = 0.0,
    slippage: float = 0.0,
    freq: str = "1D",
    # --- debug ---
    debug: bool = False,
    debug_show_examples: int = 5,
) -> vbt.Portfolio:
    required = {"open", "high", "low", "close"}
    missing = required - set(ohlcv.keys())
    if missing:
        raise ValueError(f"ohlcv missing keys: {sorted(missing)}")

    close = ohlcv["close"].copy()
    open_ = ohlcv["open"].reindex_like(close).copy()
    high = ohlcv["high"].reindex_like(close).copy()
    low = ohlcv["low"].reindex_like(close).copy()

    if horizon < 1:
        raise ValueError("horizon must be >= 1")
    if weighting not in ("equal", "proba"):
        raise ValueError("weighting must be 'equal' or 'proba'")

    if not isinstance(proba_last_class.index, pd.MultiIndex) or proba_last_class.index.nlevels != 2:
        raise TypeError("proba_last_class must be a Series with MultiIndex (ticker, earnings_ts).")

    idx = proba_last_class.index
    tick_name = idx.names[0] or "ticker"
    ts_name = idx.names[1] or "earnings_ts"

    preds_df = (
        proba_last_class.rename("proba")
        .reset_index()
        .rename(columns={tick_name: "ticker", ts_name: "earnings_ts"})
        .assign(earnings_ts=lambda d: pd.to_datetime(d["earnings_ts"]).dt.normalize())
        .pivot(index="earnings_ts", columns="ticker", values="proba")
        .sort_index()
        .reindex(index=close.index, columns=close.columns)
    )

    # --- debug: basic alignment and NaNs ---
    _dbg("==== DEBUG: inputs ====", debug)
    _dbg(f"[index] close.index: {close.index.min()} -> {close.index.max()}  n={len(close.index)}", debug)
    _dbg(f"[cols ] close.columns n={len(close.columns)}", debug)
    _df_nan_report(close, "close", debug)
    _df_nan_report(open_, "open", debug)
    _df_nan_report(high, "high", debug)
    _df_nan_report(low, "low", debug)
    _df_nan_report(preds_df, "preds_df(reindexed)", debug)

    # If close is super sparse, that alone can cause NaN stats
    if debug:
        usable_price_days = close.notna().any(axis=1).sum()
        _dbg(f"[close] days with any price data: {int(usable_price_days)}/{len(close)}", debug)

    def keep_topn_each_row(mat: pd.DataFrame, n: int) -> pd.DataFrame:
        def _row_topn(row: pd.Series) -> pd.Series:
            nn = row.notna().sum()
            if nn == 0 or nn <= n:
                return row
            keep_cols = row.nlargest(n).index
            return row.where(row.index.isin(keep_cols))
        return mat.apply(_row_topn, axis=1)

    def keep_bottomn_each_row(mat: pd.DataFrame, n: int) -> pd.DataFrame:
        def _row_bottomn(row: pd.Series) -> pd.Series:
            nn = row.notna().sum()
            if nn == 0 or nn <= n:
                return row
            keep_cols = row.nsmallest(n).index
            return row.where(row.index.isin(keep_cols))
        return mat.apply(_row_bottomn, axis=1)

    # --- LONG candidates ---
    long_cand = preds_df.copy()
    if min_proba_long is not None:
        long_cand = long_cand.where(long_cand >= float(min_proba_long))
    if top_n_long is not None:
        long_cand = keep_topn_each_row(long_cand, int(top_n_long))
    long_entries = long_cand.notna()

    # --- SHORT candidates ---
    short_cand = preds_df.copy()
    if max_proba_short is not None:
        short_cand = short_cand.where(short_cand <= float(max_proba_short))
    if top_n_short is not None:
        short_cand = keep_bottomn_each_row(short_cand, int(top_n_short))
    short_entries = short_cand.notna()

    # drop both sides if conflict
    both = long_entries & short_entries
    if both.to_numpy().any():
        long_entries = long_entries & ~both
        short_entries = short_entries & ~both
        long_cand = long_cand.where(long_entries)
        short_cand = short_cand.where(short_entries)

    _dbg("==== DEBUG: selection ====", debug)
    _bool_report(long_entries, "long_entries", debug)
    _bool_report(short_entries, "short_entries", debug)
    if debug and (both.to_numpy().any()):
        _dbg(f"[both] conflicts dropped: {int(both.to_numpy().sum())}", debug)

    # If entries exist on days where prices are NaN, vectorbt may drop/NaN things downstream.
    if debug:
        price_ok = close.notna()
        bad_long = long_entries & ~price_ok
        bad_short = short_entries & ~price_ok
        n_bad = int(bad_long.to_numpy().sum() + bad_short.to_numpy().sum())
        _dbg(f"[price check] entry signals on NaN close: {n_bad}", debug)
        if n_bad and debug_show_examples:
            ex = np.argwhere((bad_long | bad_short).to_numpy())[:debug_show_examples]
            for r, c in ex:
                side = "L" if bad_long.iat[r, c] else "S"
                _dbg(f"  example: {side} {close.columns[c]} on {close.index[r]} close=NaN", debug)

    # --- sizing (gross across both sides per day) ---
    if weighting == "equal":
        w_long = long_entries.astype(float)
        w_short = short_entries.astype(float)
    else:
        w_long = long_cand.where(long_entries)
        w_short = (1.0 - short_cand).where(short_entries)

    w_raw = (w_long.fillna(0.0) + w_short.fillna(0.0)).where((long_entries | short_entries), 0.0)
    w_sum = w_raw.sum(axis=1).replace(0.0, np.nan)
    size_frac = w_raw.div(w_sum, axis=0).fillna(0.0)
    size_frac = size_frac.where((long_entries | short_entries), 0.0)

    _dbg("==== DEBUG: sizing ====", debug)
    _row_sum_report(size_frac, "size_frac", debug)
    if debug:
        n_nonzero = int((size_frac.to_numpy() != 0).sum())
        _dbg(f"[size_frac] nonzero cells={n_nonzero}", debug)

    # --- time exits ---
    long_exits_time = pd.DataFrame(False, index=close.index, columns=close.columns)
    short_exits_time = pd.DataFrame(False, index=close.index, columns=close.columns)

    for r, c in np.argwhere(long_entries.to_numpy()):
        exit_r = r + horizon
        if exit_r < len(close.index):
            long_exits_time.iat[exit_r, c] = True

    for r, c in np.argwhere(short_entries.to_numpy()):
        exit_r = r + horizon
        if exit_r < len(close.index):
            short_exits_time.iat[exit_r, c] = True

    _dbg("==== DEBUG: exits ====", debug)
    _bool_report(long_exits_time, "long_exits_time", debug)
    _bool_report(short_exits_time, "short_exits_time", debug)

    # --- simulate ---
    pf = vbt.Portfolio.from_signals(
        close=close,
        open=open_,
        high=high,
        low=low,
        entries=long_entries,
        exits=long_exits_time,
        short_entries=short_entries,
        short_exits=short_exits_time,
        size=size_frac,
        size_type="percent",
        init_cash=init_cash,
        cash_sharing=True,
        fees=fees,
        slippage=slippage,
        freq=freq,
        sl_stop=stop_loss,
        tp_stop=take_profit,
        stop_entry_price="close",
    )

    # --- debug: portfolio diagnostics ---
    _dbg("==== DEBUG: portfolio ====", debug)
    if debug:
        # trades
        n_trades = int(pf.trades.count())
        n_pos = int(pf.positions.count())
        _dbg(f"[trades] count={n_trades}  [positions] count={n_pos}", debug)

        # value / returns
        value = pf.value()
        ret = pf.returns()

        _dbg(f"[value] nan={int(np.isnan(value.to_numpy()).sum())}/{value.size}", debug)
        _dbg(f"[returns] nan={int(np.isnan(ret.to_numpy()).sum())}/{ret.size}", debug)
        if value.size:
            _dbg(f"[value] first/last: {float(value.iloc[0]):.2f} -> {float(value.iloc[-1]):.2f}", debug)

        # if stats NaN, usually one of these:
        if n_trades == 0:
            _dbg("NOTE: No trades executed -> many stats will be NaN.", debug)
        elif np.all(np.isnan(ret.to_numpy())):
            _dbg("NOTE: Returns are all-NaN -> stats will be NaN (check price NaNs around trade windows).", debug)

        # show a tiny sample of trade records if any
        if n_trades and debug_show_examples:
            rec = pf.trades.records_readable
            _dbg(f"[trades sample]\n{rec.head(debug_show_examples).to_string(index=False)}", debug)

    return pf

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import vectorbt as vbt


def simulate_earnings_long_vbt(
    proba_last_class: pd.Series,
    ohlcv: dict[str, pd.DataFrame],
    *,
    init_cash: float = 100_000.0,
    stop_loss: float | None = 0.05,  # 5% SL from entry
    take_profit: float | None = 0.10,  # 10% TP from entry
    horizon: int = 5,  # exit at CLOSE after N trading days
    min_proba: float | None = None,
    top_n: int | None = None,
    weighting: str = "proba",  # "proba" or "equal"
    fees: float = 0.0,
    slippage: float = 0.0,
    freq: str = "1D",
) -> vbt.Portfolio:
    """Simulate long entries on earnings dates using daily OHLCV.

    Assumptions / mechanics:
    - Entry is executed at the CLOSE on the (aligned) earnings date.
    - SL/TP are evaluated starting from the next bar (t+1) using OHLC.
    - Time exit is executed at the CLOSE after `horizon` trading days.
    - Position sizing is computed daily across that day's entries using `size_type='percent'`
      and `cash_sharing=True`.

    Inputs:
    - proba_last_class: Series with MultiIndex (ticker, earnings_ts) and values = probability.
    - ohlcv: dict with keys {'open','high','low','close'} each a DataFrame [dates x tickers].

    Notes on alignment:
    - Your OHLCV index is daily (midnight) tz-naive. Earnings timestamps often contain
      intraday times (e.g., 16:00). We normalize earnings_ts to midnight so they match
      the trading calendar exactly.
    """

    # --- validate OHLCV ---
    required = {"open", "high", "low", "close"}
    missing = required - set(ohlcv.keys())
    if missing:
        raise ValueError(f"ohlcv missing keys: {sorted(missing)}")

    close = ohlcv["close"].copy()
    open_ = ohlcv["open"].reindex_like(close).copy()
    high = ohlcv["high"].reindex_like(close).copy()
    low = ohlcv["low"].reindex_like(close).copy()

    # --- validate preds index ---
    if not isinstance(proba_last_class.index, pd.MultiIndex) or proba_last_class.index.nlevels != 2:
        raise TypeError("proba_last_class must be a Series with MultiIndex (ticker, earnings_ts).")

    idx = proba_last_class.index
    tick_name = idx.names[0] or "ticker"
    ts_name = idx.names[1] or "earnings_ts"

    # --- reshape predictions to [dates x tickers] ---
    # Normalize earnings_ts to midnight so it matches close.index exactly.
    preds_df = (
        proba_last_class.rename("proba")
        .reset_index()
        .rename(columns={tick_name: "ticker", ts_name: "earnings_ts"})
        .assign(earnings_ts=lambda d: pd.to_datetime(d["earnings_ts"]).dt.normalize())
        .pivot(index="earnings_ts", columns="ticker", values="proba")
        .sort_index()
    )

    # align to trading calendar/universe (exact match on dates and tickers)
    preds_df = preds_df.reindex(index=close.index, columns=close.columns)

    # --- selection ---
    candidates = preds_df
    if min_proba is not None:
        candidates = candidates.where(candidates >= float(min_proba))

    if top_n is not None:

        def keep_topn(row: pd.Series) -> pd.Series:
            nn = row.notna().sum()
            if nn == 0 or nn <= top_n:
                return row
            keep_cols = row.nlargest(top_n).index
            return row.where(row.index.isin(keep_cols))

        candidates = candidates.apply(keep_topn, axis=1)

    entries = candidates.notna()

    # --- sizing: per-day allocation fractions (0..1), only applied on entry bars ---
    if weighting not in ("equal", "proba"):
        raise ValueError("weighting must be 'equal' or 'proba'")

    if weighting == "equal":
        w_raw = entries.astype(float)
    else:
        w_raw = candidates.where(entries)

    w_sum = w_raw.sum(axis=1).replace(0.0, np.nan)
    size_frac = w_raw.div(w_sum, axis=0).fillna(0.0)
    size_frac = size_frac.where(entries, 0.0)

    # --- time exits at CLOSE on bar + horizon ---
    if horizon < 1:
        raise ValueError("horizon must be >= 1")

    exits_time = pd.DataFrame(False, index=close.index, columns=close.columns)
    entry_rc = np.argwhere(entries.to_numpy())

    for r, c in entry_rc:
        exit_r = r + horizon
        if exit_r < exits_time.shape[0]:
            exits_time.iat[exit_r, c] = True

    # --- simulate ---
    pf = vbt.Portfolio.from_signals(
        close=close,
        open=open_,
        high=high,
        low=low,
        entries=entries,
        exits=exits_time,
        size=size_frac,
        size_type="percent",
        init_cash=init_cash,
        cash_sharing=True,
        fees=fees,
        slippage=slippage,
        freq=freq,
        sl_stop=stop_loss,
        tp_stop=take_profit,
        stop_entry_price="close",  # buy at close of earnings day
    )

    return pf

def trade_pnl_like_preds(pf, proba_last_class: pd.Series) -> pd.Series:
    """Return per-trade PnL indexed like `proba_last_class`.

    Output:
    - Series indexed by MultiIndex (ticker, earnings_ts) matching the exact ordering of
      `proba_last_class.index`.
    - If a trade did not occur for a given (ticker, earnings_ts), value will be NaN.

    Important alignment detail:
    - Your OHLCV index is daily tz-naive at midnight. To avoid mismatches between
      trade entry timestamps (from vectorbt) and prediction timestamps (often intraday),
      we normalize BOTH to midnight before reindexing.
    """

    rec = pf.trades.records_readable.copy()

    # --- ticker + entry timestamp extraction (version-tolerant) ---
    ticker = rec["Column"] if "Column" in rec.columns else pf.wrapper.columns[rec["col"].to_numpy()]

    if "Entry Timestamp" in rec.columns:
        entry_ts = pd.to_datetime(rec["Entry Timestamp"])
    else:
        entry_ts = pd.to_datetime(pf.wrapper.index[rec["entry_idx"].to_numpy()])

    # Normalize to midnight to match daily close.index convention
    entry_ts = pd.to_datetime(entry_ts).dt.normalize()

    # --- pnl extraction (version-tolerant) ---
    if "PnL" in rec.columns:
        pnl = rec["PnL"].astype(float)
    elif "pnl" in rec.columns:
        pnl = rec["pnl"].astype(float)
    else:
        pnl = pd.Series(pf.trades.pnl, index=rec.index).astype(float)

    out = (
        pd.Series(
            pnl.to_numpy(),
            index=pd.MultiIndex.from_arrays([ticker, entry_ts], names=proba_last_class.index.names),
            name="trade_pnl",
        )
        .groupby(level=[0, 1])
        .sum()
    )

    # --- normalize preds index the SAME way, then map back to original ordering ---
    idx = proba_last_class.index
    lvl0 = idx.get_level_values(0)
    lvl1 = pd.to_datetime(idx.get_level_values(1)).normalize()
    idx_norm = pd.MultiIndex.from_arrays([lvl0, lvl1], names=idx.names)

    # Reindex to normalized index (for matching), then restore original index labels/order
    return out.reindex(idx_norm).set_axis(idx)

In [ ]:
import joblib
from sklearn.ensemble import VotingClassifier

def save_model(model, path):
    """
    Saves the VotingClassifier to the specified path.
    """
    try:
        # compress=3 is a good balance between speed and file size
        joblib.dump(model, path, compress=3)
        print(f"Model successfully saved to: {path}")
    except Exception as e:
        print(f"Error saving model: {e}")

def load_model(path):
    """
    Loads the VotingClassifier from the specified path.
    """
    try:
        model = joblib.load(path)
        print("Model loaded successfully.")
        return model
    except Exception as e:
        print(f"Error loading model: {e}")
        return None



from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

#close, high, low, open_, volume=get_ohlcv(ds['ticker'].unique().tolist(),"2011-01-01")
#ds['event_day']=pd.to_datetime(whole['earnings_ts']).dt.normalize()
def meta_cvs(X,y,primary_model=LGBMClassifier(verbose=-1),meta_model=make_pipeline(SimpleImputer(fill_value=-999),LogisticRegression()),ds=ds,close=close,
             high=high,low=low,open=open_,tp=0.07,sl=0.03,primary_cvs=False,horizon=5):

  if primary_cvs:
    print(cvs(X.fillna(-999),y_new,primary_model))

  X=X.replace({np.inf: np.nan,-np.inf: np.nan})

  X_meta,y_meta=run_primary_plus_meta(
          X.fillna(-999), y, ds, close,open_,high,low,earnings_tickers,
          primary_model=primary_model,tp=tp,sl=sl,
          score_mode=True,horizon=horizon
      )

  print(X_meta.shape,y_meta.shape,close.shape)

  X_meta=X_meta.replace({np.inf: np.nan,-np.inf: np.nan})

  score = cvs(X_meta.fillna(-999), y_meta, meta_model,return_scores=True)

  return score

# Example Usage:
# model_path = 'my_voting_classifier.joblib'
# save_model(voting_clf, model_path)
# loaded_clf = load_model(model_path)

def get_feat_imps(X,y_new):

  model=LGBMClassifier(verbose=-1)
  model.fit(X,y_new)

  return pd.Series(model.feature_importances_,index=X.columns).sort_values()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time
import warnings
warnings.filterwarnings("ignore")

primary_fit=LGBMClassifier(verbose=-1)
meta_model=LGBMClassifier(verbose=-1)

#primary_fit=load_model('primary_stacking_model_TP0.07_SL0.03_HORIZON3.joblib')
#meta_model=load_model('meta_stacking_model_TP0.07_SL0.03_HORIZON3.joblib')

while True:

  close, high, low, open_, volume=get_ohlcv(earnings_tickers,"2025-01-01")
  cost_series=close.iloc[-1]
  earnings_tickers=cost_series[cost_series>10].index.tolist()
  print(len(earnings_tickers),'how many')

  out,res=run_primary_plus_meta(
      X.fillna(-999),y_new,ds,close,
      open_,high,low,
      earnings_tickers,
      primary_model=primary_fit,
      horizon=best_params['horizon'],
      meta_model=meta_model,
      sl=best_params['sl'],
      tp=best_params['tp']
  )


  # Later, inference only:
  '''
  out, test_ds, X_live, X_meta_live = infer_primary_plus_meta(
      primary_fit=primary_fit,
      meta_fit=meta_model,
      tickers=earnings_tickers,
      feature_cols=X.columns,
      #meta_feature_cols=X_meta.columns,
      anchors=ANCHORS,
  )
  '''

    # parameters (match what you used in run_primary_plus_meta)
  tp = best_params['tp']
  sl = best_params['sl']

  out['size'] = (out['p_primary'] * 2 - 1) * out['p_meta']

  # long candidates
  long_df = out.loc[(out['p_meta'] > 0.5) & (out['p_primary'] > 0.5)].copy()

  #long_df=long_df.sort_values('size').iloc[-10:]
  long_df['size']=long_df['p_primary']/abs(long_df['p_primary']).sum()

  # dollars sizing
  long_df['dollars'] = long_df['size'] * 50_000

  #entry_px = test_ds.set_index('ticker')
  entry_px=close.iloc[-1]

  # attach prices
  #long_df['entry'] = entry_px.reindex(long_df.index)['px']
  long_df['entry'] = entry_px.reindex(long_df.index)

  long_df['tp_price'] = long_df['entry'] * (1 + tp)
  long_df['sl_price'] = long_df['entry'] * (1 - sl)

  # optional: shares (integer) based on dollars / entry
  long_df['shares'] = (long_df['dollars'] / long_df['entry']).fillna(0).astype(int)

  # optional: order notionals
  long_df['notional'] = long_df['shares'] * long_df['entry']

  # optional: round to cents
  long_df[['entry', 'tp_price', 'sl_price']] = long_df[['entry', 'tp_price', 'sl_price']].round(2)

  print(long_df[['p_primary','p_meta','size','dollars','shares','entry','tp_price','sl_price']]\
        .sort_values('size').iloc[-10:])

  rebalance_to_targets(long_df)

  #break

  time.sleep(30)

'''
# ---- probabilities to size from (must be indexed by ticker) ----
p_cal = out.loc[long_list, "p_meta"]     # Series[ticker] in (0,1)

# ---- dollars for ONLY those tickers ----
w, dollars, shares, diag = size_from_probs(
    close=close,
    p_cal=p_cal,
    initial_capital=60_000,
    min_prob=0.5,   # optional; you already gated, but fine to keep
)
'''

In [ ]:
import json
import time
import pandas as pd

from alpaca.trading.client import TradingClient
from alpaca.trading.enums import OrderSide, OrderClass, TimeInForce, QueryOrderStatus
from alpaca.trading.requests import (
    MarketOrderRequest,
    LimitOrderRequest,
    TakeProfitRequest,
    StopLossRequest,
    GetOrdersRequest,
)
from alpaca.common.exceptions import APIError

# ---- helpers ----

_TERMINAL = {"filled", "canceled", "rejected", "expired"}

def _positions_map(trading: TradingClient) -> dict[str, float]:
    return {str(p.symbol): float(p.qty) for p in trading.get_all_positions()}

def _parse_apierror_payload(e: APIError) -> dict:
    try:
        return json.loads(e.args[0])
    except Exception:
        try:
            return json.loads(str(e))
        except Exception:
            return {}

def _get_symbol_orders(trading: TradingClient, symbol: str, *, limit: int = 500):
    req = GetOrdersRequest(
        status=QueryOrderStatus.ALL,
        symbols=[symbol],
        nested=True,   # helps surface bracket legs on some accounts
        limit=limit,
    )
    return trading.get_orders(filter=req)

def _iter_order_tree(order):
    yield order
    for leg in (getattr(order, "legs", None) or []):
        yield leg

def _side(o) -> str:
    return str(getattr(o, "side", "")).lower()

def _status(o) -> str:
    return str(getattr(o, "status", "")).lower()

def _is_live(o) -> bool:
    return _status(o) not in _TERMINAL

def _cancel_ids(trading: TradingClient, ids: list[str]) -> int:
    n = 0
    for oid in ids:
        try:
            trading.cancel_order_by_id(oid)
            n += 1
        except Exception:
            pass
    return n

def _cancel_all_live_sells_for_symbol(trading: TradingClient, symbol: str) -> int:
    n = 0
    for parent in _get_symbol_orders(trading, symbol):
        for o in _iter_order_tree(parent):
            if _is_live(o) and _side(o) == "sell":
                try:
                    trading.cancel_order_by_id(o.id)
                    n += 1
                except Exception:
                    pass
    return n

def _wait_no_live_sells(trading: TradingClient, symbol: str, *, timeout_s: float = 12.0, poll_s: float = 0.25) -> bool:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        live_sell = False
        for parent in _get_symbol_orders(trading, symbol):
            for o in _iter_order_tree(parent):
                if _is_live(o) and _side(o) == "sell":
                    live_sell = True
                    break
            if live_sell:
                break
        if not live_sell:
            return True
        time.sleep(poll_s)
    return False

def _is_insufficient_available(payload: dict) -> bool:
    return payload.get("code") == 40310000 and "insufficient qty available" in str(payload.get("message", "")).lower()

def _is_trading_halt_market_reject(payload: dict) -> bool:
    msg = str(payload.get("message", "")).lower()
    return payload.get("code") == 42210000 and "trading halt" in msg and "limit order" in msg

# ---- main ----

def rebalance_to_targets(
    df_targets: pd.DataFrame,
    *,
    target_qty_col: str = "shares",
    take_profit_col: str = "tp_price",
    stop_loss_col: str = "sl_price",
    use_market_entries: bool = True,
    entry_price_col: str = "entry",
    tif: TimeInForce = TimeInForce.GTC,
    paper: bool = True,

    # behavior switches
    cancel_blocking_orders: bool = True,
    halt_fallback_to_limit: bool = True,
    halt_limit_price_col: str = "entry",  # use your entry column as limit on halts
):
    required = {target_qty_col, take_profit_col, stop_loss_col}
    missing = sorted(required - set(df_targets.columns))
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # net duplicates by ticker
    df = df_targets.copy()
    df.index = df.index.astype(str)
    df = (
        df.sort_index()
          .groupby(level=0)
          .agg({
              target_qty_col: "sum",
              take_profit_col: "last",
              stop_loss_col: "last",
              **({entry_price_col: "last"} if entry_price_col in df.columns else {}),
          })
    )

    trading = TradingClient(api_key, api_secret, paper=paper)
    current = _positions_map(trading)
    results = []

    for symbol, row in df.iterrows():
        target_qty = float(row[target_qty_col])
        cur_qty = float(current.get(symbol, 0.0))
        delta = target_qty - cur_qty

        if abs(delta) < 1e-9:
            results.append({"symbol": symbol, "cur_qty": cur_qty, "target_qty": target_qty, "delta": 0.0, "action": "none"})
            continue

        # -------- SELL delta (reduce) --------
        if delta < 0:
            sell_qty = abs(delta)

            def submit_sell_market():
                return trading.submit_order(
                    MarketOrderRequest(
                        symbol=symbol,
                        qty=sell_qty,
                        side=OrderSide.SELL,
                        time_in_force=TimeInForce.DAY,
                    )
                )

            def submit_sell_limit(limit_px: float):
                return trading.submit_order(
                    LimitOrderRequest(
                        symbol=symbol,
                        qty=sell_qty,
                        side=OrderSide.SELL,
                        limit_price=limit_px,
                        time_in_force=TimeInForce.DAY,
                    )
                )

            try:
                order = submit_sell_market()
                results.append({"symbol": symbol, "action": "sell_delta", "order_id": getattr(order, "id", None), "status": getattr(order, "status", None)})
                continue

            except APIError as e:
                payload = _parse_apierror_payload(e)

                # trading halt -> place limit instead (uses your entry column as the limit price)
                if halt_fallback_to_limit and _is_trading_halt_market_reject(payload):
                    if halt_limit_price_col not in row or pd.isna(row.get(halt_limit_price_col, None)):
                        results.append({"symbol": symbol, "action": "skip_sell_halt_no_limit_price", "error": payload})
                        continue
                    order = submit_sell_limit(float(row[halt_limit_price_col]))
                    results.append({"symbol": symbol, "action": "sell_delta_limit_halt", "limit_price": float(row[halt_limit_price_col]),
                                    "order_id": getattr(order, "id", None), "status": getattr(order, "status", None)})
                    continue

                # insufficient available -> cancel blockers then retry once
                if cancel_blocking_orders and _is_insufficient_available(payload):
                    related = payload.get("related_orders") or []
                    canceled_related = _cancel_ids(trading, related)

                    # fallback: if related_orders incomplete, cancel any remaining live sells for symbol
                    canceled_fallback = _cancel_all_live_sells_for_symbol(trading, symbol)

                    _wait_no_live_sells(trading, symbol, timeout_s=15.0)

                    try:
                        order = submit_sell_market()
                        results.append({"symbol": symbol, "action": "sell_delta_after_cancel",
                                        "canceled_related": canceled_related, "canceled_fallback": canceled_fallback,
                                        "order_id": getattr(order, "id", None), "status": getattr(order, "status", None)})
                        continue
                    except APIError as e2:
                        results.append({"symbol": symbol, "action": "skip_sell_still_blocked",
                                        "canceled_related": canceled_related, "canceled_fallback": canceled_fallback,
                                        "error": _parse_apierror_payload(e2)})
                        continue

                raise

        # -------- BUY delta (increase) with bracket --------
        if delta > 0:
            tp = float(row[take_profit_col])
            sl = float(row[stop_loss_col])

            # if you want “always limit entries”, set use_market_entries=False
            if use_market_entries:
                req = MarketOrderRequest(
                    symbol=symbol,
                    qty=delta,
                    side=OrderSide.BUY,
                    time_in_force=tif,
                    order_class=OrderClass.BRACKET,
                    take_profit=TakeProfitRequest(limit_price=tp),
                    stop_loss=StopLossRequest(stop_price=sl),
                )
            else:
                req = LimitOrderRequest(
                    symbol=symbol,
                    qty=delta,
                    side=OrderSide.BUY,
                    limit_price=float(row[entry_price_col]),
                    time_in_force=tif,
                    order_class=OrderClass.BRACKET,
                    take_profit=TakeProfitRequest(limit_price=tp),
                    stop_loss=StopLossRequest(stop_price=sl),
                )

            order = trading.submit_order(req)
            results.append({"symbol": symbol, "action": "buy_delta_bracket",
                            "order_id": getattr(order, "id", None), "status": getattr(order, "status", None)})

    return pd.DataFrame(results).set_index("symbol")

In [ ]:
# pip install alpaca-py
import os
import pandas as pd

from alpaca.trading.client import TradingClient
from alpaca.trading.enums import OrderSide, OrderClass, TimeInForce
from alpaca.trading.requests import (
    MarketOrderRequest,
    LimitOrderRequest,
    TakeProfitRequest,
    StopLossRequest,
)

def submit_brackets_from_df(
    df: pd.DataFrame,
    *,
    qty_col: str = "qty",
    entry_price_col: str = "entry_price",      # used if you want LIMIT entries; can be None for market entries
    take_profit_col: str = "take_profit",
    stop_loss_col: str = "stop_loss",
    use_market_entries: bool = True,
    tif: TimeInForce = TimeInForce.GTC,
    paper: bool = True,
):
    """
    Assumes ticker symbol is df.index (e.g., df.index = ["AAPL","MSFT",...])
    Expects prices already in df columns (tp/sl). Sends a BRACKET order per row.
    """
    required = {qty_col, take_profit_col, stop_loss_col}
    if not required.issubset(df.columns):
        missing = sorted(required - set(df.columns))
        raise ValueError(f"Missing required columns: {missing}")

    if (not use_market_entries) and (entry_price_col not in df.columns):
        raise ValueError(f"use_market_entries=False requires '{entry_price_col}' column")

    # creds: set these env vars (recommended)

    # paper base_url is handled by alpaca-py via paper=True
    trading = TradingClient(api_key, api_secret, paper=paper)

    results = []
    for symbol, row in df.iterrows():
        qty = float(row[qty_col])
        tp = float(row[take_profit_col])
        sl = float(row[stop_loss_col])

        if qty <= 0:
            raise ValueError(f"{symbol}: qty must be > 0 (got {qty})")

        # Build the parent order (entry) + attached legs
        common_kwargs = dict(
            symbol=str(symbol),
            qty=qty,
            side=OrderSide.BUY,
            time_in_force=tif,
            order_class=OrderClass.BRACKET,
            take_profit=TakeProfitRequest(limit_price=tp),
            stop_loss=StopLossRequest(stop_price=sl),
        )

        if use_market_entries:
            req = MarketOrderRequest(**common_kwargs)
        else:
            entry_price = float(row[entry_price_col])
            req = LimitOrderRequest(limit_price=entry_price, **common_kwargs)

        order = trading.submit_order(req)
        results.append(
            {
                "symbol": str(symbol),
                "submitted_order_id": getattr(order, "id", None),
                "qty": qty,
                "entry_type": "market" if use_market_entries else "limit",
                "take_profit": tp,
                "stop_loss": sl,
                "status": getattr(order, "status", None),
            }
        )

    return pd.DataFrame(results).set_index("symbol")


# ---- Example usage ----
# df index = ticker symbols
# df columns must include: qty, take_profit, stop_loss
# optionally entry_price if use_market_entries=False
#
# df = pd.DataFrame(
#     {"qty":[10,5], "take_profit":[210.0, 420.0], "stop_loss":[190.0, 380.0]},
#     index=["AAPL","MSFT"]
# )
#
out = submit_brackets_from_df(long_df, use_market_entries=True, paper=True)
# print(out)

In [ ]:
!pip install alpaca-py

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

def size_from_probs(
    close: pd.DataFrame,          # date x ticker
    p_cal: pd.Series,             # index=ticker, values in (0,1)
    asof: pd.Timestamp | None = None,
    horizon_days: int = 5,
    vol_lookback: int = 20,
    initial_capital: float = 100_000.0,

    # risk controls
    kelly_scale: float = 0.25,    # shrink Kelly (0.1–0.5 typical)
    max_gross: float = 1.0,       # max sum of weights (long-only)
    max_weight: float = 0.10,     # per-name cap
    min_prob: float = 0.55,       # only size if p >= this
    eps: float = 1e-6
):
    """
    Returns:
      weights: fraction of equity per ticker (sum <= max_gross)
      dollars: dollar allocation per ticker
      shares: share allocation per ticker at asof close
      diagnostics: df with p, z, sigma_h, raw_kelly
    """
    if asof is None:
        asof = close.index[-1]
    else:
        asof = pd.Timestamp(asof)

    # Align tickers present in both
    tickers = [t for t in p_cal.index if t in close.columns]
    p = p_cal.loc[tickers].astype(float)

    # price at asof
    px = close.loc[:asof, tickers].iloc[-1]

    # vol estimate (daily) -> horizon vol
    ret = close[tickers].pct_change()
    sigma_d = ret.loc[:asof].rolling(vol_lookback).std().iloc[-1]
    sigma_h = sigma_d * np.sqrt(horizon_days)

    # convert prob -> z-score (edge proxy)
    p_clip = p.clip(eps, 1 - eps)
    z = pd.Series(norm.ppf(p_clip), index=tickers)

    # long-only: ignore p < 0.5; enforce min_prob gate
    z = z.where(p >= min_prob, other=0.0)

    # Kelly-like fraction (normal approx): f ~ mu / var = (sigma_h*z) / sigma_h^2 = z / sigma_h
    raw = z / (sigma_h.replace(0.0, np.nan))

    # shrink + clean
    raw = (kelly_scale * raw).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    raw = raw.clip(lower=0.0)  # long-only

    # per-name cap
    w = raw.clip(upper=max_weight)

    # gross cap (sum weights <= max_gross)
    gross = w.sum()
    if gross > max_gross and gross > 0:
        w = w * (max_gross / gross)

    dollars = w * initial_capital
    shares = (dollars / px).fillna(0.0)

    diagnostics = pd.DataFrame({
        "p_cal": p,
        "z": z,
        "sigma_d": sigma_d,
        "sigma_h": sigma_h,
        "raw_kelly": raw,
        "weight": w,
        "dollars": dollars,
        "price": px,
        "shares": shares
    }).sort_values("weight", ascending=False)

    return w, dollars, shares, diagnostics

import pandas as pd
import yfinance as yf

def get_px(
    tickers,
    start="1998-01-01",
    end=None,
    use_adj_close=True,
    ffill_limit=3,
):
    """
    Returns a DataFrame:
      index  = trading days
      columns = tickers
      values  = prices
    """

    raw = yf.download(
        tickers,
        start=start,
        end=end,
        interval="1d",
        auto_adjust=False,
        group_by="ticker",
        progress=False,
        threads=True,
        ignore_tz=True,
    )

    # --- extract price panel ---
    if isinstance(raw.columns, pd.MultiIndex):
        fields = raw.columns.get_level_values(1).unique()
        px_field = "Adj Close" if (use_adj_close and "Adj Close" in fields) else "Close"
        px = raw.xs(px_field, axis=1, level=1)
    else:
        px_field = "Adj Close" if (use_adj_close and "Adj Close" in raw.columns) else "Close"
        px = raw[[px_field]]
        px.columns = tickers[:1]

    px = px.sort_index()
    px = px.ffill(limit=ffill_limit)

    return px

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt
from lightgbm import LGBMClassifier

def cv_predict_proba_purged(model, X, y, cv):
    """Return OOF proba aligned to X.index using provided (purged) CV splits."""
    oof = pd.Series(index=X.index, dtype=float)

    # PurgedTimeSeriesSplit usually yields (train_idx, test_idx) as integer positions
    for tr_idx, te_idx in cv.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_te = X.iloc[te_idx]

        #m = model.__class__(**model.get_params())
        model.fit(X_tr, y_tr)
        oof.iloc[te_idx] = model.predict_proba(X_te)[:, 1]

    return oof


import numpy as np
import pandas as pd

def make_event_signal_matrices(
    index_df,
    px_close,
    p_primary,
    horizon=5,
    side_threshold=0.5,
    debug=False,
    map_to_next_trading_day=False,
    sample_n=10,
):
    """
    index_df: DataFrame aligned to p_primary.index with column 'event_day'
             Index should be (ticker, earnings_ts)
    px_close: wide DataFrame (dates x tickers)
    p_primary: Series indexed by (ticker, earnings_ts)
    """

    # ---- debug header ----
    if debug:
        print("\n=== make_event_signal_matrices DEBUG ===")
        print("index_df.index.names:", index_df.index.names)
        print("p_primary.index.names:", p_primary.index.names)
        print("px_close.index.name(s):", getattr(px_close.index, "names", None))
        print("px_close.shape:", px_close.shape)
        print("px_close.index dtype:", px_close.index.dtype)
        print("px_close.index min/max:", px_close.index.min(), px_close.index.max())
        print("px_close.columns (n):", len(px_close.columns))

    # Ensure px_close has DatetimeIndex
    px_close = px_close.copy()
    px_close.index = pd.to_datetime(px_close.index)

    # Ensure event_day exists and align

    idx = p_primary.index
    tmp = index_df.reindex(idx)[['event_day']].copy()

    tmp['p_primary'] = p_primary.reindex(tmp.index).values

    tmp = tmp.dropna(subset=['event_day', 'p_primary'])

    if debug:
        print("\n-- after reindex --")
        print("tmp.shape:", tmp.shape)
        print("tmp.event_day null count:", tmp['event_day'].isna().sum())
        print("tmp head:")
        print(tmp.head(3))

    # Convert event_day to datetime
    tmp['event_day_raw'] = tmp['event_day']  # keep original for debugging
    tmp['event_day'] = pd.to_datetime(tmp['event_day'], errors='coerce')

    if debug:
        bad_parse = tmp['event_day'].isna().sum()
        print("\n-- datetime parse --")
        print("bad event_day parse count:", bad_parse)
        if bad_parse > 0:
            print("examples of unparseable event_day:")
            print(tmp.loc[tmp['event_day'].isna(), 'event_day_raw'].head(sample_n))

    # Normalize to date (midnight)
    tmp['event_day_norm'] = tmp['event_day'].dt.normalize()

    # Also normalize px index (common cause: px index has time component or is tz-aware)
    px_norm = pd.to_datetime(px_close.index).normalize()
    px_close.index = px_norm

    if debug:
        print("\n-- normalization --")
        print("tmp event_day_norm min/max:",
              tmp['event_day_norm'].min(), tmp['event_day_norm'].max())
        print("px_close.index (normalized) min/max:",
              px_close.index.min(), px_close.index.max())

        # Check if px index has duplicates after normalize
        dup = px_close.index.duplicated().sum()
        if dup:
            print("WARNING: px_close index has duplicates after normalize:", dup)

    # Drop rows with missing event_day
    tmp = tmp.dropna(subset=['event_day_norm'])
    if debug:
        print("\n-- after dropping NaN event_day_norm --")
        print("tmp.shape:", tmp.shape)

    # Check overlap BEFORE filtering
    in_px = tmp['event_day_norm'].isin(px_close.index)
    if debug:
        print("\n-- overlap check --")
        print("events on px dates:", int(in_px.sum()), " / ", len(in_px))

        if len(in_px) > 0:
            # show a few that miss
            miss = tmp.loc[~in_px, 'event_day_norm'].dropna()
            if len(miss) > 0:
                print("sample missing event_day_norm values:")
                print(miss.head(sample_n).to_list())

            # common reason: off by one day due to after-hours; check next/prev trading day existence
            miss_days = tmp.loc[~in_px, 'event_day_norm'].dropna()
            if len(miss_days) > 0:
                # count weekends quickly
                weekend = miss_days.dt.weekday >= 5
                print("missing that fall on weekend:", int(weekend.sum()), "/", len(miss_days))

                # check if +1 day would match
                plus1 = (miss_days + pd.Timedelta(days=1)).isin(px_close.index)
                minus1 = (miss_days - pd.Timedelta(days=1)).isin(px_close.index)
                print("missing where +1 day matches px:", int(plus1.sum()), "/", len(miss_days))
                print("missing where -1 day matches px:", int(minus1.sum()), "/", len(miss_days))

    # Optionally map to next available trading day instead of dropping
    if map_to_next_trading_day:
        px_dates = px_close.index.values  # datetime64[ns]
        ed = tmp['event_day_norm'].values.astype('datetime64[ns]')

        pos = np.searchsorted(px_dates, ed, side='left')
        pos = np.clip(pos, 0, len(px_dates) - 1)
        mapped = px_dates[pos]

        if debug:
            mapped_in = pd.Series(mapped).isin(px_close.index).mean()
            print("\n-- mapping to next trading day --")
            print("mapped coverage ratio:", mapped_in)

        tmp['event_day_use'] = mapped
    else:
        tmp['event_day_use'] = tmp['event_day_norm']

    # Now filter to those in px index
    tmp = tmp[tmp['event_day_use'].isin(px_close.index)]

    if debug:
        print("\n-- final kept events --")
        print("tmp.shape:", tmp.shape)
        if tmp.shape[0] == 0:
            # helpful diagnostics: range mismatch
            print("NO EVENTS KEPT.")
            print("event_day_norm unique count:", tmp['event_day_norm'].nunique() if 'event_day_norm' in tmp else None)
            # show original pre-filter ranges from earlier captured vars (recompute quickly)
            # (we still have px range above)
        else:
            print("sample kept rows:")
            print(tmp.head(3)[['event_day_raw', 'event_day', 'event_day_norm', 'event_day_use', 'p_primary']])

    if tmp.empty:
        raise ValueError("No events matched to px_close index after normalization/filtering.")

    dates = px_close.index
    tickers = px_close.columns

    entries_long  = pd.DataFrame(False, index=dates, columns=tickers)
    entries_short = pd.DataFrame(False, index=dates, columns=tickers)

    long_mask = tmp['p_primary'] >= side_threshold

    for (tkr, ets), row in tmp.iterrows():
        d = row['event_day_use']
        if tkr not in tickers:
            continue
        if long_mask.loc[(tkr, ets)]:
            entries_long.loc[d, tkr] = True
        else:
            entries_short.loc[d, tkr] = True

    exits_long  = entries_long.vbt.signals.fshift(horizon, fill_value=False)
    exits_short = entries_short.vbt.signals.fshift(horizon, fill_value=False)

    # return tmp with chosen event day
    out_tmp = tmp.copy()
    out_tmp['event_day'] = out_tmp['event_day_use']
    out_tmp = out_tmp.drop(columns=['event_day_use'])

    return out_tmp, entries_long, exits_long, entries_short, exits_short



def attach_returns_to_events(tmp_events, trades, px_close):
    """
    Map vectorbt trades back to your event rows via (ticker, event_day).
    tmp_events: DataFrame indexed by (ticker, earnings_ts) with event_day + p_primary
    trades: vectorbt readable records
    """
    # trades has 'Column' (ticker), 'Entry Timestamp' etc in readable form
    t = trades.copy()
    t['event_day'] = pd.to_datetime(t['Entry Timestamp']).dt.normalize()
    t['ticker'] = t['Column'].astype(str)
    t['trade_ret'] = t['Return'].astype(float)

    # aggregate if multiple trades somehow occur same day per ticker (shouldn't, but safe)
    trade_map = (
        t.groupby(['ticker', 'event_day'], as_index=False)['trade_ret']
         .mean()
    )

    out = tmp_events.copy()
    out['event_day'] = pd.to_datetime(out['event_day']).dt.normalize()
    out = out.reset_index()
    out = out.merge(trade_map, on=['ticker', 'event_day'], how='left')
    out = out.set_index(['ticker', 'earnings_ts'])

    return out


import vectorbt as vbt

def vectorbt_trade_returns_gapaware(
    open_df, high_df, low_df, close_df,
    entries_long, exits_long, entries_short, exits_short,
    tp=0.02, sl=0.01, init_cash=1.0
):
    # Enums live in the docs under portfolio.enums (StopEntryPrice / StopExitPrice) :contentReference[oaicite:2]{index=2}
    StopEntryPrice = vbt.portfolio.enums.StopEntryPrice
    StopExitPrice  = vbt.portfolio.enums.StopExitPrice

    pf = vbt.Portfolio.from_signals(
        close=close_df,
        open=open_df,
        high=high_df,
        low=low_df,

        entries=entries_long,
        exits=exits_long,
        short_entries=entries_short,
        short_exits=exits_short,

        # Percent stops relative to entry
        sl_stop=sl,
        tp_stop=tp,

        # Entry at close of signal bar (event day close)
        stop_entry_price=StopEntryPrice.Close,

        # If gapped through stop/TP, exit at next bar open; otherwise intra-bar via OHLC.
        # StopMarket applies slippage; StopLimit does not :contentReference[oaicite:3]{index=3}
        stop_exit_price=StopExitPrice.StopMarket,

        init_cash=init_cash,
        fees=0.0,
        slippage=0.0,
        freq="1D",
        direction="both"
    )

    return pf.trades.records_readable.copy()

def build_meta_dataset(X, p_primary, trade_ret, min_abs_ret=0.0):
    """
    X: base features (aligned to events)
    p_primary: primary OOF proba (aligned)
    trade_ret: realized trade returns aligned to events (positive means profit in predicted direction)
    """
    df = X.copy()
    df['p_primary'] = p_primary

    # meta label
    meta_y = (trade_ret > 0).astype(int)

    # Optional: drop tiny outcomes (noise)
    if min_abs_ret > 0:
        keep = trade_ret.abs() >= min_abs_ret
        df = df.loc[keep]
        meta_y = meta_y.loc[keep]
        trade_ret = trade_ret.loc[keep]

    return df, meta_y

def run_primary_plus_meta(
    X, y, data_with_event_day, px_close,px_open,px_high,px_low, tickers,
    horizon=5,
    primary_model=None,
    meta_model=None,
    side_threshold=0.5,
    meta_min_abs_ret=0.0,
    gap=121,
    anchors=ANCHORS,
    score_mode=False,
    sl=0.032,
    tp=0.032,
    save_meta_model_path=None,
    long_only=True,
):
    if primary_model is None:
        primary_model = LGBMClassifier(verbose=-1)
    if meta_model is None:
        meta_model = make_pipeline(SimpleImputer(fill_value=-999),LogisticRegression())

    X=X.replace({np.inf: np.nan,-np.inf: np.nan})
    X=X.copy().fillna(-999)

    px_close = px_close.copy()
    px_close.index = pd.to_datetime(px_close.index).normalize()

    if score_mode:
      print('Number of mismatched tickers:',set(ds['ticker'])-set(px_close))

    # --- Purged CV object (you already do this) ---
    dates = pd.Series(X.index.get_level_values('earnings_ts'))
    cv = PurgedTimeSeriesSplit(dates=dates, gap=gap)

    # --- 1) OOF primary proba ---
    p_oof = cv_predict_proba_purged(primary_model, X, y, cv)
    p_oof = p_oof.loc[X.index]

    # --- build event-day frame aligned to X ---
    d = data_with_event_day.copy()

    d['earnings_ts'] = pd.to_datetime(d['earnings_ts'])
    d['event_day'] = pd.to_datetime(d['event_day'])

    d = d.set_index(['ticker', 'earnings_ts']).sort_index()

    # align to X
    d = d.reindex(X.index)

    print("aligned d.index.names:", d.index.names)
    print("aligned event_day non-null:", d['event_day'].notna().sum(), "/", len(d))
    print("aligned p_oof non-null:", p_oof.notna().sum(), "/", len(p_oof))

    # optional: show a few missing rows
    missing = d['event_day'].isna()
    print("missing event_day rows:", missing.sum())
    if missing.any():
        print("sample missing keys:", d.index[missing][:10].to_list())

    idx_df = d[['event_day']].copy()

    # --- 2) vectorbt realized returns for each OOF “trade” ---
    # --- build event-day frame aligned to X ---
    d = data_with_event_day.copy()
    d['earnings_ts'] = pd.to_datetime(d['earnings_ts'])
    d['event_day'] = pd.to_datetime(d['event_day'])

    d = d.set_index(['ticker', 'earnings_ts']).sort_index()
    d = d.reindex(X.index)

    idx_df = d[['event_day']].copy()
    print("PASSING idx_df.index.names:", idx_df.index.names)  # should be ['ticker','earnings_ts']

    tmp_events, el, xl, es, xs = make_event_signal_matrices(
        index_df=idx_df,
        px_close=px_close,
        p_primary=p_oof,
        horizon=horizon,
        side_threshold=side_threshold,
        debug=False,
    )

    if long_only:
      es[:] = False
      xs[:] = False

    trades = vectorbt_trade_returns_gapaware(
              open_df=px_open,
              high_df=px_high,
              low_df=px_low,
              close_df=px_close,
              entries_long=el, exits_long=xl,
              entries_short=es, exits_short=xs,
              tp=tp, sl=sl
          )

    print(trades.shape,'trades shape')
    print(len(tmp_events),'events lengths')

    events_with_ret = attach_returns_to_events(tmp_events, trades, px_close)

    print(events_with_ret.dropna(subset=['trade_ret']).shape,'shape after dropping nans')

    # ================= CONSISTENCY ASSERTIONS =================

    # 1) Entry day must match labeler rule
    def _labeler_event_day(earnings_ts, trade_days):
        ts = earnings_ts.to_numpy(dtype="datetime64[ns]")
        pos = np.searchsorted(trade_days, ts, side="right") - 1

        out = np.full(len(ts), np.datetime64("NaT"), dtype="datetime64[ns]")
        ok = pos >= 0
        out[ok] = trade_days[pos[ok]]
        return pd.to_datetime(out).normalize()


    trade_days = px_close.index.values.astype("datetime64[ns]")

    dcheck = tmp_events.reset_index().copy()
    dcheck["earnings_ts"] = pd.to_datetime(dcheck["earnings_ts"])

    dcheck["event_day_labeler"] = _labeler_event_day(
        dcheck["earnings_ts"],
        trade_days
    )

    dcheck["event_day_used"] = pd.to_datetime(dcheck["event_day"]).dt.normalize()

    bad_day = dcheck["event_day_labeler"] != dcheck["event_day_used"]

    assert not bad_day.any(), (
        "Event day mismatch with labeler mapping:\n"
        + dcheck.loc[bad_day,
            ["ticker", "earnings_ts", "event_day_used", "event_day_labeler"]
          ].head(10).to_string()
    )


    # 2) No short trades allowed in meta (long-only invariant)
    if long_only:
        assert (es.values == 0).all(), "Short entries present while long_only=True"
        assert (xs.values == 0).all(), "Short exits present while long_only=True"


    # 3) Meta label must agree with horizon sign when no stop triggered
    #    (soft check: allow tiny tolerance)
    merged = events_with_ret.copy()

    merged["meta_y"] = (merged["trade_ret"] > 0).astype(int)

    # Compare with your original y if available
    if "y_new" in locals():
        y_aligned = y_new.reindex(merged.index)

        agree = (merged["meta_y"] == y_aligned) | y_aligned.isna()

        disagree_rate = 1 - agree.mean()

        assert disagree_rate < 0.01, (   # 1% tolerance
            f"Meta/primary label disagreement too high: {disagree_rate:.2%}"
        )

    # ==========================================================


    trade_ret = events_with_ret['trade_ret'].dropna()
    print(trade_ret.shape,'trade ret shape')

    common_idx = X.index.intersection(trade_ret.index)

    print(len(common_idx),'len common')

    trade_ret = trade_ret.loc[common_idx]
    X_meta_base = X.loc[common_idx]
    p_oof_ok = p_oof.loc[common_idx]
    y_ok = y.loc[common_idx]

    print(y_ok.shape,'initial y')

    long_only_mask = p_oof_ok >= side_threshold
    trade_ret = trade_ret.loc[long_only_mask]
    X_meta_base = X_meta_base.loc[long_only_mask]
    p_oof_ok = p_oof_ok.loc[long_only_mask]
    y_ok = y_ok.loc[long_only_mask]

    print(y_ok.shape,'y after long filter')

    # --- 3) Meta dataset (features + p_primary) -> meta label ---
    X_meta, y_meta = build_meta_dataset(
        X=X_meta_base,
        p_primary=p_oof_ok,
        trade_ret=trade_ret,
        min_abs_ret=meta_min_abs_ret
    )

    if score_mode:
      return X_meta,y_meta

    # --- 4) Fit models on full calibration ---
    primary_fit = primary_model
    primary_fit.fit(X, y)

    meta_fit = meta_model
    meta_fit.fit(X_meta, y_meta)

    if save_meta_model_path:
        pd.concat([X_meta,y_meta],axis=1).to_csv(f'meta_dataset_{primary_model.__str__()[:16]}.csv')
        save_model(meta_fit,save_meta_model_path)
        save_model(primary_fit,save_meta_model_path.replace('meta','primary'))

    # --- 5) Build synthetic “today earnings” test set (you already have this function) ---
    test_ds = build_synthetic_earnings_test_dataset(
        tickers=tickers,
        asof=None,
        start="1998-01-01",
        anchor=anchors if anchors is not None else ["SPY"],
        min_hist_days=200,
    )

    # Align columns
    X_live = test_ds[X.columns].copy()

    # --- 6) Live primary proba ---
    p_primary_live = pd.Series(
        primary_fit.predict_proba(X_live)[:, 1],
        index=test_ds['ticker'].values
    )

    # --- 7) Live meta proba (use same feature recipe: X + p_primary) ---
    X_meta_live = X_live.copy()
    X_meta_live['p_primary'] = p_primary_live.values

    p_meta_live = pd.Series(
        meta_fit.predict_proba(X_meta_live[X_meta.columns])[:, 1],
        index=test_ds['ticker'].values
    )

    # --- 8) One simple position sizing rule ---
    # signed conviction in [-1, 1] times probability of profit:
    signed = (p_primary_live * 2.0 - 1.0)   # long if >0, short if <0
    size = signed * p_meta_live            # scale by meta prob

    out = pd.DataFrame({
        'p_primary': p_primary_live,
        'p_meta': p_meta_live,
        'size': size
    }).sort_values('size', ascending=False)

    return out, {
        'p_oof': p_oof,
        'trade_ret': trade_ret,
        'X_meta': X_meta,
        'y_meta': y_meta
    }

In [ ]:
import numpy as np
import pandas as pd

def infer_primary_plus_meta(
    primary_fit,
    meta_fit,
    tickers,
    feature_cols,
    meta_feature_cols=None,
    anchors=None,
    asof=None,
    start="1998-01-01",
    min_hist_days=200,
):
    """
    Pure inference:
      1) build synthetic test dataset
      2) compute primary proba
      3) compute meta proba (using same feature recipe)
      4) output sizing signal

    Returns:
      out_df, test_ds, X_live, X_meta_live
    """

    if meta_feature_cols is None:
        meta_feature_cols = list(feature_cols) + ["p_primary"]

    # --- 1) Build synthetic test set ---
    test_ds = build_synthetic_earnings_test_dataset(
        tickers=tickers,
        asof=asof,
        start=start,
        anchor=anchors if anchors is not None else ["SPY"],
        min_hist_days=min_hist_days,
    )

    # --- 2) Align base features exactly ---
    X_live = test_ds.reindex(columns=feature_cols).copy()

    # handle inf/nan similar to your training path
    X_live = X_live.replace([np.inf, -np.inf], np.nan).fillna(-999)

    # --- 3) Primary proba ---
    p_primary_live = pd.Series(
        primary_fit.predict_proba(X_live)[:, 1],
        index=test_ds["ticker"].values
    )

    # --- 4) Meta proba ---
    X_meta_live = X_live.copy()
    X_meta_live["p_primary"] = p_primary_live.values

    X_meta_live = X_meta_live.reindex(columns=meta_feature_cols).copy()
    X_meta_live = X_meta_live.replace([np.inf, -np.inf], np.nan).fillna(-999)

    p_meta_live = pd.Series(
        meta_fit.predict_proba(X_meta_live)[:, 1],
        index=test_ds["ticker"].values
    )

    # --- 5) Simple sizing rule ---
    signed = (p_primary_live * 2.0 - 1.0)   # [-1, 1]
    size = signed * p_meta_live

    out = (
        pd.DataFrame({"p_primary": p_primary_live, "p_meta": p_meta_live, "size": size})
        .sort_values("size", ascending=False)
    )

    return out, test_ds, X_live, X_meta_live